In [ ]:
# ============================================================
# IMPROVED CSE-CIC-IDS2018 — OFFICIAL DOWNLOAD
# ============================================================

import os
import shutil

DATASET_URL = (
    "https://intrusion-detection.distrinet-research.be/"
    "CNS2022/Datasets/CSECICIDS2018_improved.zip"
)

ZIP_PATH = "/content/CSECICIDS2018_improved.zip"

# ------------------------------------------------------------
# 1. Check available Colab storage
# ------------------------------------------------------------

total, used, free = shutil.disk_usage("/content")

print("Colab local storage:")
print(f"  Total: {total / 1024**3:,.2f} GB")
print(f"  Used:  {used / 1024**3:,.2f} GB")
print(f"  Free:  {free / 1024**3:,.2f} GB")

MINIMUM_FREE_GB = 15

if free / 1024**3 < MINIMUM_FREE_GB:
    raise RuntimeError(
        f"Only {free / 1024**3:.2f} GB is free. "
        "At least 15 GB is recommended for the compressed archive."
    )

# ------------------------------------------------------------
# 2. Download with resume support
# ------------------------------------------------------------

print("\nDownloading the official improved dataset...")

!wget -c \
    --progress=bar:force:noscroll \
    --timeout=60 \
    --tries=20 \
    -O "$ZIP_PATH" \
    "$DATASET_URL"

# ------------------------------------------------------------
# 3. Basic file verification
# ------------------------------------------------------------

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Download did not create: {ZIP_PATH}"
    )

file_size_gb = os.path.getsize(ZIP_PATH) / 1024**3

print("\nDownloaded archive:")
print(f"  Path: {ZIP_PATH}")
print(f"  Size: {file_size_gb:,.2f} GB")

!ls -lh "$ZIP_PATH"

Colab local storage:
  Total: 225.83 GB
  Used:  0.08 GB
  Free:  225.74 GB

--2026-11-02 16:40:08--  https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CSECICIDS2018_improved.zip
Resolving intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)... 134.58.40.205
Connecting to intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)|134.58.40.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10426851729 (9.7G) [application/zip]
Saving to: ‘/content/CSECICIDS2018_improved.zip’

/content/CSECICIDS2 100%[===================>]   9.71G  13.8MB/s    in 11m 13s 

2026-11-02 16:51:21 (14.8 MB/s) - ‘/content/CSECICIDS2018_improved.zip’ saved [10426851729/10426851729]


Downloaded archive:
  Path: /content/CSECICIDS2018_improved.zip
  Size: 9.71 GB
-rw-r--r-- 1 root root 9.8G Apr  3  2023 /content/CSECICIDS2018_improved.zip


In [ ]:
# ============================================================
# IMPROVED CSE-CIC-IDS2018 — ARCHIVE MANIFEST
# ============================================================

import os
import zipfile
import pandas as pd

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Archive not found: {ZIP_PATH}"
    )

if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError(
        "The downloaded file is not recognized as a valid ZIP archive."
    )

with zipfile.ZipFile(ZIP_PATH) as archive:

    members = [
        info
        for info in archive.infolist()
        if not info.is_dir()
    ]

    manifest = pd.DataFrame(
        [
            {
                "Filename": info.filename,
                "Compressed_MB": (
                    info.compress_size / 1024**2
                ),
                "Uncompressed_MB": (
                    info.file_size / 1024**2
                ),
                "Extension": os.path.splitext(
                    info.filename
                )[1].lower(),
            }
            for info in members
        ]
    )

csv_manifest = (
    manifest[
        manifest["Extension"].eq(".csv")
    ]
    .copy()
    .sort_values("Filename")
    .reset_index(drop=True)
)

print("=" * 110)
print("ARCHIVE SUMMARY")
print("=" * 110)

print(f"All files: {len(manifest):,}")
print(f"CSV files: {len(csv_manifest):,}")

print(
    f"Total compressed size: "
    f"{manifest['Compressed_MB'].sum() / 1024:,.2f} GB"
)

print(
    f"Total uncompressed size: "
    f"{manifest['Uncompressed_MB'].sum() / 1024:,.2f} GB"
)

print("\nCSV files:")
print(
    csv_manifest[
        [
            "Filename",
            "Compressed_MB",
            "Uncompressed_MB",
        ]
    ]
    .round(2)
    .to_string(index=False)
)

ARCHIVE SUMMARY
All files: 10
CSV files: 10
Total compressed size: 9.71 GB
Total uncompressed size: 33.57 GB

CSV files:
                Filename  Compressed_MB  Uncompressed_MB
   Friday-02-03-2018.csv        1020.76          3518.90
   Friday-16-02-2018.csv        1130.81          4016.63
   Friday-23-02-2018.csv         957.20          3251.43
 Thursday-01-03-2018.csv        1064.40          3631.97
 Thursday-15-02-2018.csv         835.06          2853.90
 Thursday-22-02-2018.csv         974.34          3312.90
  Tuesday-20-02-2018.csv         949.38          3272.29
Wednesday-14-02-2018.csv         887.63          3111.44
Wednesday-21-02-2018.csv        1062.68          3769.58
Wednesday-28-02-2018.csv        1061.56          3635.62


In [ ]:
# ============================================================
# IMPROVED CSE-CIC-IDS2018 — FROZEN-PROTOCOL SCHEMA CHECK
# ============================================================

import io
import re
import zipfile
import pandas as pd

# ------------------------------------------------------------
# 1. Frozen-protocol requirements
# ------------------------------------------------------------

REQUIRED_COMPONENTS = {
    "Timestamp": [
        "Timestamp",
        "Flow Timestamp",
        "DateTime",
        "Datetime",
        "Start Time",
    ],

    "Source IP": [
        "Src IP",
        "Source IP",
        "SrcIP",
        "src_ip",
    ],

    "Destination IP": [
        "Dst IP",
        "Destination IP",
        "DstIP",
        "dst_ip",
    ],

    "Flow IAT Max": [
        "Flow IAT Max",
        "Flow_IAT_Max",
    ],

    "Avg Bwd Segment Size": [
        "Bwd Segment Size Avg",
        "Avg Bwd Segment Size",
        "Avg_Bwd_Segment_Size",
        "Bwd_Segment_Size_Avg",
    ],

    "Down/Up Ratio": [
        "Down/Up Ratio",
        "Down Up Ratio",
        "Down_Up_Ratio",
    ],

    "Label": [
        "Label",
        "label",
        "Class",
        "Attack Label",
    ],

    # Needed to apply the improved dataset's ground-truth policy.
    "Attempted Category": [
        "Attempted Category",
        "Attempted_Category",
        "AttemptedCategory",
    ],
}


def normalize_column_name(name):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(name).strip().lower(),
    )


def match_required_column(columns, aliases):
    normalized_columns = {
        normalize_column_name(column): column
        for column in columns
    }

    for alias in aliases:
        normalized_alias = normalize_column_name(
            alias
        )

        if normalized_alias in normalized_columns:
            return normalized_columns[
                normalized_alias
            ]

    return None


def read_csv_header_from_zip(
    archive,
    filename,
    nrows=3,
):
    # Primary encoding
    try:
        with archive.open(filename) as stream:
            sample = pd.read_csv(
                stream,
                nrows=nrows,
                low_memory=False,
            )

        return sample

    except UnicodeDecodeError:
        with archive.open(filename) as stream:
            sample = pd.read_csv(
                stream,
                nrows=nrows,
                low_memory=False,
                encoding="latin-1",
            )

        return sample


# ------------------------------------------------------------
# 2. Inspect every CSV separately
# ------------------------------------------------------------

schema_records = []
file_schemas = {}
sample_rows = {}

with zipfile.ZipFile(ZIP_PATH) as archive:

    csv_files = sorted(
        [
            info.filename
            for info in archive.infolist()
            if (
                not info.is_dir()
                and info.filename.lower().endswith(".csv")
            )
        ]
    )

    if not csv_files:
        raise ValueError(
            "No CSV files were found inside the archive."
        )

    print(f"CSV files found: {len(csv_files):,}\n")

    for file_number, filename in enumerate(
        csv_files,
        start=1,
    ):
        print(
            f"[{file_number:02d}/{len(csv_files):02d}] "
            f"Inspecting {filename}"
        )

        sample = read_csv_header_from_zip(
            archive,
            filename,
            nrows=3,
        )

        columns = sample.columns.tolist()

        file_schemas[filename] = columns
        sample_rows[filename] = sample

        for requirement, aliases in (
            REQUIRED_COMPONENTS.items()
        ):
            matched = match_required_column(
                columns,
                aliases,
            )

            schema_records.append({
                "Filename": filename,
                "Required_Component": requirement,
                "Status": (
                    "FOUND"
                    if matched is not None
                    else "MISSING"
                ),
                "Matched_Column": (
                    matched
                    if matched is not None
                    else ""
                ),
                "Total_Columns": len(columns),
            })

schema_check = pd.DataFrame(
    schema_records
)

# ------------------------------------------------------------
# 3. Per-file compatibility
# ------------------------------------------------------------

file_compatibility = (
    schema_check
    .groupby(
        "Filename",
        observed=True,
    )
    .agg(
        Required_Components=(
            "Required_Component",
            "size",
        ),
        Found_Components=(
            "Status",
            lambda values: int(
                values.eq("FOUND").sum()
            ),
        ),
        Missing_Components=(
            "Required_Component",
            lambda values: ", ".join(
                schema_check.loc[
                    values.index[
                        schema_check.loc[
                            values.index,
                            "Status",
                        ].eq("MISSING")
                    ],
                    "Required_Component",
                ].tolist()
            ),
        ),
        Total_Columns=(
            "Total_Columns",
            "first",
        ),
    )
    .reset_index()
)

file_compatibility["Fully_Compatible"] = (
    file_compatibility[
        "Found_Components"
    ]
    ==
    file_compatibility[
        "Required_Components"
    ]
)

print("\n" + "=" * 110)
print("PER-FILE FROZEN-PROTOCOL COMPATIBILITY")
print("=" * 110)

print(
    file_compatibility
    .to_string(index=False)
)

# ------------------------------------------------------------
# 4. Detailed missing-component report
# ------------------------------------------------------------

missing_report = (
    schema_check[
        schema_check["Status"].eq("MISSING")
    ]
    .copy()
)

print("\n" + "=" * 110)
print("MISSING COMPONENTS")
print("=" * 110)

if missing_report.empty:
    print(
        "PASS: Every CSV contains all frozen-protocol "
        "and ground-truth-control columns."
    )
else:
    print(
        missing_report[
            [
                "Filename",
                "Required_Component",
            ]
        ]
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Show matched names across the archive
# ------------------------------------------------------------

matched_names = (
    schema_check[
        schema_check["Status"].eq("FOUND")
    ]
    .groupby(
        "Required_Component",
        observed=True,
    )["Matched_Column"]
    .agg(
        lambda values: sorted(
            set(values)
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("MATCHED COLUMN NAMES")
print("=" * 110)

print(
    matched_names.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 6. Final decision
# ------------------------------------------------------------

all_files_compatible = bool(
    file_compatibility[
        "Fully_Compatible"
    ].all()
)

print("\n" + "=" * 110)
print("SCHEMA DECISION")
print("=" * 110)

if all_files_compatible:
    print(
        "FULL COMPATIBILITY: all CSV files support the "
        "frozen structural and behavioral protocol."
    )
else:
    print(
        "PARTIAL COMPATIBILITY: do not run the protocol yet."
    )
    print(
        "Inspect missing columns and determine whether they "
        "are naming differences or genuinely absent."
    )

CSV files found: 10

[01/10] Inspecting Friday-02-03-2018.csv
[02/10] Inspecting Friday-16-02-2018.csv
[03/10] Inspecting Friday-23-02-2018.csv
[04/10] Inspecting Thursday-01-03-2018.csv
[05/10] Inspecting Thursday-15-02-2018.csv
[06/10] Inspecting Thursday-22-02-2018.csv
[07/10] Inspecting Tuesday-20-02-2018.csv
[08/10] Inspecting Wednesday-14-02-2018.csv
[09/10] Inspecting Wednesday-21-02-2018.csv
[10/10] Inspecting Wednesday-28-02-2018.csv

PER-FILE FROZEN-PROTOCOL COMPATIBILITY
                Filename  Required_Components  Found_Components Missing_Components  Total_Columns  Fully_Compatible
   Friday-02-03-2018.csv                    8                 8                                91              True
   Friday-16-02-2018.csv                    8                 8                                91              True
   Friday-23-02-2018.csv                    8                 8                                91              True
 Thursday-01-03-2018.csv                    8    

In [ ]:
# ============================================================
# IMPROVED CSE-CIC-IDS2018
# TEMPORAL BASELINE AND GROUND-TRUTH SCAN
#
# Primary policy:
#   Attempted flows are treated as BENIGN.
#
# Sensitivity policy:
#   Any non-BENIGN Label is treated as ATTACK.u
#
# Candidate calibration:
#   The earliest continuous sequence of attack-free hourly
#   windows strictly preceding the first effective attack hour.
# ============================================================

import os
import zipfile
from collections import Counter

import numpy as np
import pandas as pd

ZIP_PATH = "/content/CSECICIDS2018_improved.zip"

USE_COLUMNS = [
    "Timestamp",
    "Label",
    "Attempted Category",
]

CHUNK_SIZE = 400_000

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(ZIP_PATH)

print("=" * 110)
print("CSE-CIC-IDS2018 — TEMPORAL BASELINE SCAN")
print("=" * 110)

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def normalize_text(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.upper()
    )


def attempted_mask(series):
    """
    Values equivalent to -1 mean 'not attempted'.
    Any other non-empty value denotes an attempted category.
    """
    normalized = normalize_text(series)

    no_attempt_values = {
        "",
        "-1",
        "-1.0",
        "NAN",
        "NONE",
        "NULL",
        "BENIGN",
    }

    return ~normalized.isin(no_attempt_values)


# ------------------------------------------------------------
# 2. Scan CSV files in chronological filename order
# ------------------------------------------------------------

hourly_parts = []
file_rows = []
label_counter = Counter()
attempted_counter = Counter()

with zipfile.ZipFile(ZIP_PATH) as archive:

    csv_files = [
        info.filename
        for info in archive.infolist()
        if (
            not info.is_dir()
            and info.filename.lower().endswith(".csv")
        )
    ]

    # Parse date from filename for chronological ordering
    def filename_date(filename):
        basename = os.path.basename(filename)

        match = pd.Series([basename]).str.extract(
            r"(\d{2}-\d{2}-\d{4})"
        ).iloc[0, 0]

        return pd.to_datetime(
            match,
            format="%d-%m-%Y",
            errors="coerce",
        )

    csv_files = sorted(
        csv_files,
        key=filename_date,
    )

    print(f"CSV files: {len(csv_files)}\n")

    for file_number, filename in enumerate(
        csv_files,
        start=1,
    ):
        print(
            f"[{file_number:02d}/{len(csv_files):02d}] "
            f"{filename}"
        )

        file_flow_count = 0
        file_primary_attack_count = 0
        file_sensitivity_attack_count = 0
        file_attempted_count = 0
        file_timestamp_failures = 0
        file_start = None
        file_end = None

        with archive.open(filename) as stream:

            reader = pd.read_csv(
                stream,
                usecols=USE_COLUMNS,
                chunksize=CHUNK_SIZE,
                low_memory=False,
            )

            for chunk in reader:

                timestamps = pd.to_datetime(
                    chunk["Timestamp"],
                    errors="coerce",
                    dayfirst=True,
                    format="mixed",
                )

                labels = normalize_text(
                    chunk["Label"]
                )

                is_attempted = attempted_mask(
                    chunk["Attempted Category"]
                )

                # Sensitivity policy:
                # all non-BENIGN labels count as attacks.
                attack_sensitivity = (
                    labels.ne("BENIGN")
                )

                # Primary policy:
                # attempted events are treated as benign.
                attack_primary = (
                    attack_sensitivity
                    &
                    ~is_attempted
                )

                valid_timestamp = (
                    timestamps.notna()
                )

                file_timestamp_failures += int(
                    (~valid_timestamp).sum()
                )

                if valid_timestamp.any():
                    chunk_start = timestamps[
                        valid_timestamp
                    ].min()

                    chunk_end = timestamps[
                        valid_timestamp
                    ].max()

                    file_start = (
                        chunk_start
                        if file_start is None
                        else min(
                            file_start,
                            chunk_start,
                        )
                    )

                    file_end = (
                        chunk_end
                        if file_end is None
                        else max(
                            file_end,
                            chunk_end,
                        )
                    )

                file_flow_count += len(chunk)
                file_primary_attack_count += int(
                    attack_primary.sum()
                )
                file_sensitivity_attack_count += int(
                    attack_sensitivity.sum()
                )
                file_attempted_count += int(
                    is_attempted.sum()
                )

                label_counter.update(
                    labels.value_counts().to_dict()
                )

                attempted_counter.update(
                    normalize_text(
                        chunk["Attempted Category"]
                    )
                    .value_counts()
                    .to_dict()
                )

                scan_chunk = pd.DataFrame({
                    "_hour": timestamps.dt.floor("1h"),
                    "_attack_primary": attack_primary.astype(
                        np.int8
                    ),
                    "_attack_sensitivity": attack_sensitivity.astype(
                        np.int8
                    ),
                    "_attempted": is_attempted.astype(
                        np.int8
                    ),
                })

                scan_chunk = scan_chunk.dropna(
                    subset=["_hour"]
                )

                chunk_hourly = (
                    scan_chunk
                    .groupby(
                        "_hour",
                        observed=True,
                    )
                    .agg(
                        Flows=(
                            "_attack_primary",
                            "size",
                        ),
                        Primary_Attack_Flows=(
                            "_attack_primary",
                            "sum",
                        ),
                        Sensitivity_Attack_Flows=(
                            "_attack_sensitivity",
                            "sum",
                        ),
                        Attempted_Flows=(
                            "_attempted",
                            "sum",
                        ),
                    )
                    .reset_index()
                )

                chunk_hourly["_source_file"] = filename
                hourly_parts.append(chunk_hourly)

        file_rows.append({
            "Filename": filename,
            "Start_Time": file_start,
            "End_Time": file_end,
            "Flows": file_flow_count,
            "Primary_Attack_Flows": (
                file_primary_attack_count
            ),
            "Sensitivity_Attack_Flows": (
                file_sensitivity_attack_count
            ),
            "Attempted_Flows": file_attempted_count,
            "Timestamp_Failures": (
                file_timestamp_failures
            ),
        })

        print(
            f"  Flows={file_flow_count:,} | "
            f"Primary attacks={file_primary_attack_count:,} | "
            f"Attempted={file_attempted_count:,}"
        )

# ------------------------------------------------------------
# 3. Consolidate hourly counts across chunks
# ------------------------------------------------------------

hourly_scan = (
    pd.concat(
        hourly_parts,
        ignore_index=True,
    )
    .groupby(
        "_hour",
        observed=True,
    )
    .agg(
        Flows=("Flows", "sum"),
        Primary_Attack_Flows=(
            "Primary_Attack_Flows",
            "sum",
        ),
        Sensitivity_Attack_Flows=(
            "Sensitivity_Attack_Flows",
            "sum",
        ),
        Attempted_Flows=(
            "Attempted_Flows",
            "sum",
        ),
    )
    .reset_index()
    .sort_values("_hour")
    .reset_index(drop=True)
)

del hourly_parts

hourly_scan["Primary_Attack_Rate"] = (
    hourly_scan["Primary_Attack_Flows"]
    / hourly_scan["Flows"]
)

hourly_scan["Sensitivity_Attack_Rate"] = (
    hourly_scan["Sensitivity_Attack_Flows"]
    / hourly_scan["Flows"]
)

hourly_scan["Primary_Benign_Hour"] = (
    hourly_scan["Primary_Attack_Flows"]
    .eq(0)
)

hourly_scan["Sensitivity_Benign_Hour"] = (
    hourly_scan["Sensitivity_Attack_Flows"]
    .eq(0)
)

file_summary = (
    pd.DataFrame(file_rows)
    .sort_values("Start_Time")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Identify the first attack hour
# ------------------------------------------------------------

primary_attack_hours = hourly_scan[
    hourly_scan["Primary_Attack_Flows"] > 0
]

sensitivity_attack_hours = hourly_scan[
    hourly_scan["Sensitivity_Attack_Flows"] > 0
]

first_primary_attack_hour = (
    primary_attack_hours["_hour"].min()
    if not primary_attack_hours.empty
    else pd.NaT
)

first_sensitivity_attack_hour = (
    sensitivity_attack_hours["_hour"].min()
    if not sensitivity_attack_hours.empty
    else pd.NaT
)

# ------------------------------------------------------------
# 5. Select earliest continuous benign prefix
# ------------------------------------------------------------

if pd.isna(first_primary_attack_hour):
    calibration_candidate = (
        hourly_scan.copy()
    )
else:
    calibration_candidate = (
        hourly_scan[
            hourly_scan["_hour"]
            < first_primary_attack_hour
        ]
        .copy()
    )

# Keep only a continuous hourly sequence from the beginning.
if not calibration_candidate.empty:

    calibration_candidate = (
        calibration_candidate
        .sort_values("_hour")
        .reset_index(drop=True)
    )

    expected_hours = pd.date_range(
        start=calibration_candidate[
            "_hour"
        ].iloc[0],
        periods=len(calibration_candidate),
        freq="1h",
    )

    continuity = (
        calibration_candidate["_hour"]
        .reset_index(drop=True)
        .eq(pd.Series(expected_hours))
    )

    if not continuity.all():
        first_gap_position = int(
            np.flatnonzero(
                ~continuity.to_numpy()
            )[0]
        )

        calibration_candidate = (
            calibration_candidate
            .iloc[:first_gap_position]
            .copy()
        )

    calibration_candidate = (
        calibration_candidate[
            calibration_candidate[
                "Primary_Benign_Hour"
            ]
        ]
        .copy()
    )

# ------------------------------------------------------------
# 6. Summaries
# ------------------------------------------------------------

label_summary = (
    pd.DataFrame(
        label_counter.items(),
        columns=["Label", "Flows"],
    )
    .sort_values(
        "Flows",
        ascending=False,
    )
    .reset_index(drop=True)
)

attempted_summary = (
    pd.DataFrame(
        attempted_counter.items(),
        columns=[
            "Attempted_Category",
            "Flows",
        ],
    )
    .sort_values(
        "Flows",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 110)
print("PER-FILE TEMPORAL AND LABEL SUMMARY")
print("=" * 110)

print(
    file_summary.to_string(
        index=False
    )
)

print("\n" + "=" * 110)
print("FIRST ATTACK HOURS")
print("=" * 110)

print(
    "Primary policy "
    "(Attempted → BENIGN):     ",
    first_primary_attack_hour,
)

print(
    "Sensitivity policy "
    "(Label ≠ BENIGN):         ",
    first_sensitivity_attack_hour,
)

print("\n" + "=" * 110)
print("CANDIDATE CHRONOLOGICAL CALIBRATION PREFIX")
print("=" * 110)

if calibration_candidate.empty:
    print(
        "No continuous benign prefix was found "
        "before the first effective attack."
    )
else:
    calibration_start = (
        calibration_candidate["_hour"].min()
    )

    calibration_end_exclusive = (
        calibration_candidate["_hour"].max()
        + pd.Timedelta(hours=1)
    )

    print(
        f"Start:          {calibration_start}"
    )
    print(
        f"End exclusive:  {calibration_end_exclusive}"
    )
    print(
        f"Hours:          {len(calibration_candidate):,}"
    )
    print(
        f"Flows:          "
        f"{calibration_candidate['Flows'].sum():,}"
    )
    print(
        f"Primary attacks:"
        f" {calibration_candidate['Primary_Attack_Flows'].sum():,}"
    )
    print(
        f"Attempted flows:"
        f" {calibration_candidate['Attempted_Flows'].sum():,}"
    )

print("\n" + "=" * 110)
print("LABEL SUMMARY")
print("=" * 110)

print(
    label_summary.to_string(
        index=False
    )
)

print("\n" + "=" * 110)
print("ATTEMPTED-CATEGORY SUMMARY")
print("=" * 110)

print(
    attempted_summary.head(30).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 7. Save audit outputs
# ------------------------------------------------------------

hourly_scan.to_csv(
    "/content/csecic2018_hourly_ground_truth_scan.csv",
    index=False,
)

file_summary.to_csv(
    "/content/csecic2018_file_summary.csv",
    index=False,
)

calibration_candidate.to_csv(
    "/content/csecic2018_candidate_calibration_prefix.csv",
    index=False,
)

label_summary.to_csv(
    "/content/csecic2018_label_summary.csv",
    index=False,
)

attempted_summary.to_csv(
    "/content/csecic2018_attempted_category_summary.csv",
    index=False,
)

print("\nSaved:")
print(
    "/content/csecic2018_hourly_ground_truth_scan.csv"
)
print(
    "/content/csecic2018_file_summary.csv"
)
print(
    "/content/csecic2018_candidate_calibration_prefix.csv"
)
print(
    "/content/csecic2018_label_summary.csv"
)
print(
    "/content/csecic2018_attempted_category_summary.csv"
)

CSE-CIC-IDS2018 — TEMPORAL BASELINE SCAN
CSV files: 10

[01/10] Wednesday-14-02-2018.csv
  Flows=5,898,350 | Primary attacks=94,197 | Attempted=193,354
[02/10] Thursday-15-02-2018.csv
  Flows=5,410,102 | Primary attacks=31,050 | Attempted=6,581
[03/10] Friday-16-02-2018.csv
  Flows=7,390,266 | Primary attacks=1,803,160 | Attempted=105,606
[04/10] Tuesday-20-02-2018.csv
  Flows=6,054,702 | Primary attacks=290,125 | Attempted=80
[05/10] Wednesday-21-02-2018.csv
  Flows=6,962,593 | Primary attacks=1,084,023 | Attempted=171
[06/10] Thursday-22-02-2018.csv
  Flows=6,071,153 | Primary attacks=125 | Attempted=83
[07/10] Friday-23-02-2018.csv
  Flows=5,976,481 | Primary attacks=158 | Attempted=72
[08/10] Wednesday-28-02-2018.csv
  Flows=6,568,726 | Primary attacks=49,829 | Attempted=15
[09/10] Thursday-01-03-2018.csv
  Flows=6,551,401 | Primary attacks=39,834 | Attempted=13
[10/10] Friday-02-03-2018.csv
  Flows=6,311,371 | Primary attacks=142,921 | Attempted=262

PER-FILE TEMPORAL AND LABEL SU

In [ ]:
# ============================================================
# CSE-CIC-IDS2018 — FREEZE AND VERIFY CALIBRATION POLICY
#
# Primary calibration:
#   2018-02-14 12:00 <= timestamp < 2018-02-14 18:00
#   Label == BENIGN
#   Attempted Category == -1
#
# Primary evaluation truth:
#   Attempted flows are treated as BENIGN.
#
# Sensitivity truth:
#   Any non-BENIGN label is treated as ATTACK.
# ============================================================

import os
import zipfile
from collections import Counter

import pandas as pd

ZIP_PATH = "/content/CSECICIDS2018_improved.zip"

CALIBRATION_START = pd.Timestamp(
    "2018-02-14 12:00:00"
)

CALIBRATION_END = pd.Timestamp(
    "2018-02-14 18:00:00"
)

EVALUATION_START = CALIBRATION_END

USE_COLUMNS = [
    "Timestamp",
    "Label",
    "Attempted Category",
]

CHUNK_SIZE = 400_000


def normalize_text(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.upper()
    )


def is_attempted(series):
    normalized = normalize_text(series)

    non_attempt_values = {
        "-1",
        "-1.0",
        "",
        "NAN",
        "NONE",
        "NULL",
    }

    return ~normalized.isin(
        non_attempt_values
    )


if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        ZIP_PATH
    )


counts = Counter()

with zipfile.ZipFile(ZIP_PATH) as archive:

    csv_files = sorted(
        [
            info.filename
            for info in archive.infolist()
            if (
                not info.is_dir()
                and info.filename.lower().endswith(".csv")
            )
        ]
    )

    for file_number, filename in enumerate(
        csv_files,
        start=1,
    ):
        print(
            f"[{file_number:02d}/{len(csv_files):02d}] "
            f"{filename}"
        )

        with archive.open(filename) as stream:

            reader = pd.read_csv(
                stream,
                usecols=USE_COLUMNS,
                chunksize=CHUNK_SIZE,
                low_memory=False,
            )

            for chunk in reader:

                timestamp = pd.to_datetime(
                    chunk["Timestamp"],
                    errors="coerce",
                    dayfirst=True,
                    format="mixed",
                )

                label = normalize_text(
                    chunk["Label"]
                )

                attempted = is_attempted(
                    chunk["Attempted Category"]
                )

                valid_timestamp = (
                    timestamp.notna()
                )

                in_candidate_period = (
                    valid_timestamp
                    &
                    timestamp.ge(
                        CALIBRATION_START
                    )
                    &
                    timestamp.lt(
                        CALIBRATION_END
                    )
                )

                ordinary_benign = (
                    label.eq("BENIGN")
                    &
                    ~attempted
                )

                primary_calibration = (
                    in_candidate_period
                    &
                    ordinary_benign
                )

                attempted_in_candidate = (
                    in_candidate_period
                    &
                    attempted
                )

                nonbenign_in_candidate = (
                    in_candidate_period
                    &
                    label.ne("BENIGN")
                    &
                    ~attempted
                )

                strict_time_baseline = (
                    valid_timestamp
                    &
                    timestamp.ge(
                        CALIBRATION_START
                    )
                    &
                    timestamp.lt(
                        pd.Timestamp(
                            "2018-02-14 14:00:00"
                        )
                    )
                )

                evaluation = (
                    valid_timestamp
                    &
                    timestamp.ge(
                        EVALUATION_START
                    )
                )

                # Primary truth:
                # attempted flows are treated as benign.
                primary_attack = (
                    evaluation
                    &
                    label.ne("BENIGN")
                    &
                    ~attempted
                )

                # Sensitivity truth:
                # every non-BENIGN label is an attack.
                sensitivity_attack = (
                    evaluation
                    &
                    label.ne("BENIGN")
                )

                counts[
                    "all_valid_flows"
                ] += int(
                    valid_timestamp.sum()
                )

                counts[
                    "candidate_period_flows"
                ] += int(
                    in_candidate_period.sum()
                )

                counts[
                    "primary_calibration_flows"
                ] += int(
                    primary_calibration.sum()
                )

                counts[
                    "attempted_excluded_from_calibration"
                ] += int(
                    attempted_in_candidate.sum()
                )

                counts[
                    "effective_attacks_in_calibration"
                ] += int(
                    nonbenign_in_candidate.sum()
                )

                counts[
                    "strict_pre_attempt_flows"
                ] += int(
                    strict_time_baseline.sum()
                )

                counts[
                    "evaluation_flows"
                ] += int(
                    evaluation.sum()
                )

                counts[
                    "evaluation_primary_attacks"
                ] += int(
                    primary_attack.sum()
                )

                counts[
                    "evaluation_sensitivity_attacks"
                ] += int(
                    sensitivity_attack.sum()
                )


policy_summary = pd.DataFrame(
    [
        {
            "Item": key,
            "Flows": value,
        }
        for key, value in counts.items()
    ]
)

print("\n" + "=" * 105)
print("FROZEN CALIBRATION POLICY")
print("=" * 105)

print(
    f"Calibration start: "
    f"{CALIBRATION_START}"
)

print(
    f"Calibration end:   "
    f"{CALIBRATION_END} "
    f"(exclusive)"
)

print(
    f"Evaluation start:  "
    f"{EVALUATION_START}"
)

print(
    "\nCalibration eligibility:"
)

print(
    "  Label == BENIGN "
    "AND Attempted Category == -1"
)

print("\n" + "=" * 105)
print("SPLIT VERIFICATION")
print("=" * 105)

print(
    policy_summary.to_string(
        index=False
    )
)


candidate_total = counts[
    "candidate_period_flows"
]

calibration_total = counts[
    "primary_calibration_flows"
]

attempted_excluded = counts[
    "attempted_excluded_from_calibration"
]

effective_attacks = counts[
    "effective_attacks_in_calibration"
]

print("\nDerived checks:")

print(
    f"  Calibration retention: "
    f"{calibration_total / candidate_total:.4%}"
)

print(
    f"  Attempted exclusion rate: "
    f"{attempted_excluded / candidate_total:.4%}"
)

print(
    f"  Effective attacks retained "
    f"in calibration: {effective_attacks:,}"
)

if effective_attacks != 0:
    raise AssertionError(
        "The frozen calibration still contains "
        "effective attack flows."
    )

if calibration_total == 0:
    raise AssertionError(
        "No eligible calibration flows remain."
    )

print(
    "\nPASS: The primary calibration contains "
    "ordinary benign flows only."
)

print(
    "NOTE: This is label-assisted calibration, "
    "not label-free calibration."
)

policy_summary.to_csv(
    "/content/"
    "csecic2018_frozen_split_policy_summary.csv",
    index=False,
)

frozen_policy = {
    "calibration_start": (
        CALIBRATION_START
    ),
    "calibration_end": (
        CALIBRATION_END
    ),
    "evaluation_start": (
        EVALUATION_START
    ),
    "calibration_label_policy": (
        "Label == BENIGN and "
        "Attempted Category == -1"
    ),
    "primary_truth_policy": (
        "Attempted -> BENIGN"
    ),
    "sensitivity_truth_policy": (
        "Label != BENIGN -> ATTACK"
    ),
}

print(
    "\nCreated object: frozen_policy"
)

print(
    "Saved: "
    "/content/"
    "csecic2018_frozen_split_policy_summary.csv"
)

[01/10] Friday-02-03-2018.csv
[02/10] Friday-16-02-2018.csv
[03/10] Friday-23-02-2018.csv
[04/10] Thursday-01-03-2018.csv


In [ ]:
# ============================================================
# CELL 1 — CSE-CIC-IDS2018 FROZEN CALIBRATION
# ============================================================

import os
import re
import gc
import zipfile
import numpy as np
import pandas as pd

ZIP_PATH = "/content/CSECICIDS2018_improved.zip"

CAL_START = pd.Timestamp("2018-02-14 12:00:00")
CAL_END   = pd.Timestamp("2018-02-14 18:00:00")
EVAL_START = CAL_END

CHUNK_SIZE = 400_000

USE_COLUMNS = [
    "Timestamp",
    "Src IP",
    "Dst IP",
    "Flow IAT Max",
    "Bwd Segment Size Avg",
    "Down/Up Ratio",
    "Label",
    "Attempted Category",
]

FEATURES = {
    "PACKET": "Bwd Segment Size Avg",
    "TEMPORAL": "Flow IAT Max",
    "DIRECTIONAL": "Down/Up Ratio",
}

def norm_text(series):
    return series.astype(str).str.strip().str.upper()

def attempted_mask(series):
    x = norm_text(series)
    return ~x.isin({"-1", "-1.0", "", "NAN", "NONE", "NULL"})

def ip_hash(series):
    return pd.util.hash_pandas_object(
        series.astype(str).str.strip(),
        index=False,
    ).astype("uint64")

def file_date(name):
    match = re.search(r"(\d{2}-\d{2}-\d{4})", os.path.basename(name))
    return pd.to_datetime(
        match.group(1),
        format="%d-%m-%Y",
        errors="coerce",
    ) if match else pd.NaT

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(ZIP_PATH)

# ------------------------------------------------------------
# 1. Locate chronological CSV files and calibration file
# ------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH) as archive:
    csv_files = sorted(
        [
            info.filename
            for info in archive.infolist()
            if not info.is_dir()
            and info.filename.lower().endswith(".csv")
        ],
        key=file_date,
    )

calibration_files = [
    name for name in csv_files
    if file_date(name) == pd.Timestamp("2018-02-14")
]

if len(calibration_files) != 1:
    raise RuntimeError(
        f"Expected one 2018-02-14 file; found {calibration_files}"
    )

CALIBRATION_FILE = calibration_files[0]

print("Calibration file:", CALIBRATION_FILE)

# ------------------------------------------------------------
# 2. First calibration pass:
#    structural features + exact flow-level feature arrays
# ------------------------------------------------------------

cal_struct_parts = []
cal_edge_parts = []
cal_feature_arrays = {
    mechanism: []
    for mechanism in FEATURES
}

eligible_calibration_flows = 0

with zipfile.ZipFile(ZIP_PATH) as archive:
    with archive.open(CALIBRATION_FILE) as stream:

        reader = pd.read_csv(
            stream,
            usecols=USE_COLUMNS,
            chunksize=CHUNK_SIZE,
            low_memory=False,
        )

        for chunk_number, chunk in enumerate(reader, start=1):

            ts = pd.to_datetime(
                chunk["Timestamp"],
                errors="coerce",
                dayfirst=True,
                format="mixed",
            )

            labels = norm_text(chunk["Label"])
            attempted = attempted_mask(chunk["Attempted Category"])

            mask = (
                ts.notna()
                & ts.ge(CAL_START)
                & ts.lt(CAL_END)
                & labels.eq("BENIGN")
                & ~attempted
            )

            if not mask.any():
                continue

            # Keep all filtered objects on the same positional RangeIndex
            part = (
                chunk.loc[mask]
                .copy()
                .reset_index(drop=True)
            )

            part_ts = (
                ts.loc[mask]
                .reset_index(drop=True)
            )

            src = (
                part["Src IP"]
                .astype(str)
                .str.strip()
                .reset_index(drop=True)
            )

            dst = (
                part["Dst IP"]
                .astype(str)
                .str.strip()
                .reset_index(drop=True)
            )

            # Use a positional NumPy mask to prevent index alignment errors
            valid_id = (
                src.ne("")
                & dst.ne("")
                & src.str.lower().ne("nan")
                & dst.str.lower().ne("nan")
            ).to_numpy(dtype=bool)

            valid_positions = np.flatnonzero(valid_id)

            part = (
                part.iloc[valid_positions]
                .reset_index(drop=True)
            )

            part_ts = (
                part_ts.iloc[valid_positions]
                .reset_index(drop=True)
            )

            src = (
                src.iloc[valid_positions]
                .reset_index(drop=True)
            )

            dst = (
                dst.iloc[valid_positions]
                .reset_index(drop=True)
            )

            src_h = ip_hash(src)
            dst_h = ip_hash(dst)
            window_ns = (
                part_ts.dt.floor("1h")
                .astype("int64")
                .to_numpy()
            )

            eligible_calibration_flows += len(part)

            structural_chunk = pd.DataFrame({
                "src_hash": src_h.to_numpy(),
                "window_ns": window_ns,
                "dst_hash": dst_h.to_numpy(),
                "timestamp": part_ts.to_numpy(),
            })

            cal_struct_parts.append(
                structural_chunk
                .groupby(
                    ["src_hash", "window_ns"],
                    observed=True,
                )
                .agg(
                    flow_count=("dst_hash", "size"),
                    first_time=("timestamp", "min"),
                    last_time=("timestamp", "max"),
                )
                .reset_index()
            )

            cal_edge_parts.append(
                structural_chunk[
                    ["src_hash", "window_ns", "dst_hash"]
                ].drop_duplicates()
            )

            for mechanism, feature in FEATURES.items():
                values = pd.to_numeric(
                    part[feature],
                    errors="coerce",
                ).to_numpy(dtype="float64")

                values = values[np.isfinite(values)]
                cal_feature_arrays[mechanism].append(values)

            print(
                f"Chunk {chunk_number}: "
                f"eligible calibration flows={eligible_calibration_flows:,}"
            )

# ------------------------------------------------------------
# 3. Consolidate calibration structural windows
# ------------------------------------------------------------

cal_structural = (
    pd.concat(cal_struct_parts, ignore_index=True)
    .groupby(
        ["src_hash", "window_ns"],
        observed=True,
    )
    .agg(
        flow_count=("flow_count", "sum"),
        first_time=("first_time", "min"),
        last_time=("last_time", "max"),
    )
    .reset_index()
)

cal_edges = (
    pd.concat(cal_edge_parts, ignore_index=True)
    .drop_duplicates(
        ["src_hash", "window_ns", "dst_hash"]
    )
)

cal_unique_dst = (
    cal_edges
    .groupby(
        ["src_hash", "window_ns"],
        observed=True,
    )
    .size()
    .rename("unique_dst_ip")
    .reset_index()
)

cal_structural = (
    cal_structural
    .merge(
        cal_unique_dst,
        on=["src_hash", "window_ns"],
        how="left",
        validate="one_to_one",
    )
)

cal_structural["activity_duration_seconds"] = (
    pd.to_datetime(cal_structural["last_time"])
    - pd.to_datetime(cal_structural["first_time"])
).dt.total_seconds()

cal_structural["active_span_flow_rate"] = (
    cal_structural["flow_count"]
    / cal_structural["activity_duration_seconds"].clip(lower=1)
)

cal_structural["destination_repetition"] = (
    cal_structural["flow_count"]
    / cal_structural["unique_dst_ip"].replace(0, np.nan)
).fillna(0)

# Frozen structural percentiles
structural_thresholds = {
    "BV_RATE_P99": float(
        cal_structural["active_span_flow_rate"].quantile(0.99)
    ),
    "BV_COUNT_P99": float(
        cal_structural["flow_count"].quantile(0.99)
    ),
    "REP_RATIO_P99": float(
        cal_structural["destination_repetition"].quantile(0.99)
    ),
    "REP_COUNT_P50": float(
        cal_structural["flow_count"].quantile(0.50)
    ),
}

# ------------------------------------------------------------
# 4. Exact two-sided flow-level P01/P99 thresholds
# ------------------------------------------------------------

behavioral_flow_thresholds = {}

for mechanism, arrays in cal_feature_arrays.items():

    values = np.concatenate(arrays)
    values = values[np.isfinite(values)]

    behavioral_flow_thresholds[mechanism] = {
        "lower_p01": float(np.quantile(values, 0.01)),
        "upper_p99": float(np.quantile(values, 0.99)),
        "valid_flows": int(len(values)),
    }

    del values

gc.collect()

# ------------------------------------------------------------
# 5. Second calibration pass:
#    window-level outlier count/share distributions
# ------------------------------------------------------------

cal_behavior_parts = []

with zipfile.ZipFile(ZIP_PATH) as archive:
    with archive.open(CALIBRATION_FILE) as stream:

        reader = pd.read_csv(
            stream,
            usecols=USE_COLUMNS,
            chunksize=CHUNK_SIZE,
            low_memory=False,
        )

        for chunk in reader:

            ts = pd.to_datetime(
                chunk["Timestamp"],
                errors="coerce",
                dayfirst=True,
                format="mixed",
            )

            labels = norm_text(chunk["Label"])
            attempted = attempted_mask(chunk["Attempted Category"])

            mask = (
                ts.notna()
                & ts.ge(CAL_START)
                & ts.lt(CAL_END)
                & labels.eq("BENIGN")
                & ~attempted
            )

            if not mask.any():
                continue

            part = chunk.loc[mask].copy().reset_index(drop=True)
            part_ts = ts.loc[mask].reset_index(drop=True)

            src = part["Src IP"].astype(str).str.strip().reset_index(drop=True)

            valid_id = (
                src.ne("")
                & src.str.lower().ne("nan")
            )

            part = part.loc[valid_id].reset_index(drop=True)
            part_ts = part_ts.loc[valid_id].reset_index(drop=True)
            src = src.loc[valid_id].reset_index(drop=True)

            out = pd.DataFrame({
                "src_hash": ip_hash(src).to_numpy(),
                "window_ns": (
                    part_ts.dt.floor("1h")
                    .astype("int64")
                    .to_numpy()
                ),
            })

            aggregation = {}

            for mechanism, feature in FEATURES.items():

                values = pd.to_numeric(
                    part[feature],
                    errors="coerce",
                ).to_numpy(dtype="float64")

                valid = np.isfinite(values)

                thresholds = behavioral_flow_thresholds[mechanism]

                anomaly = (
                    valid
                    & (
                        (values < thresholds["lower_p01"])
                        | (values > thresholds["upper_p99"])
                    )
                )

                lower = mechanism.lower()

                out[f"{lower}_valid"] = valid.astype("int8")
                out[f"{lower}_outlier"] = anomaly.astype("int8")

                aggregation[f"{lower}_valid_count"] = (
                    f"{lower}_valid",
                    "sum",
                )
                aggregation[f"{lower}_outlier_count"] = (
                    f"{lower}_outlier",
                    "sum",
                )

            cal_behavior_parts.append(
                out
                .groupby(
                    ["src_hash", "window_ns"],
                    observed=True,
                )
                .agg(**aggregation)
                .reset_index()
            )

cal_behavior = (
    pd.concat(cal_behavior_parts, ignore_index=True)
    .groupby(
        ["src_hash", "window_ns"],
        observed=True,
    )
    .sum(numeric_only=True)
    .reset_index()
)

for mechanism in FEATURES:
    lower = mechanism.lower()

    cal_behavior[f"{lower}_outlier_share"] = (
        cal_behavior[f"{lower}_outlier_count"]
        / cal_behavior[f"{lower}_valid_count"].replace(0, np.nan)
    ).fillna(0)

behavioral_window_thresholds = {}

for mechanism in FEATURES:
    lower = mechanism.lower()

    behavioral_window_thresholds[mechanism] = {
        "count_p95": float(
            cal_behavior[f"{lower}_outlier_count"].quantile(0.95)
        ),
        "share_p95": float(
            cal_behavior[f"{lower}_outlier_share"].quantile(0.95)
        ),
    }

# ------------------------------------------------------------
# 6. Display and save frozen thresholds
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("CALIBRATION SUMMARY")
print("=" * 110)

print(f"Eligible calibration flows: {eligible_calibration_flows:,}")
print(f"Calibration source-windows: {len(cal_structural):,}")

print("\nStructural thresholds:")
for key, value in structural_thresholds.items():
    print(f"  {key}: {value:,.8f}")

print("\nBehavioral flow thresholds:")
for mechanism, values in behavioral_flow_thresholds.items():
    print(
        f"  {mechanism}: "
        f"P01={values['lower_p01']:,.8f}, "
        f"P99={values['upper_p99']:,.8f}, "
        f"N={values['valid_flows']:,}"
    )

print("\nBehavioral window thresholds:")
for mechanism, values in behavioral_window_thresholds.items():
    print(
        f"  {mechanism}: "
        f"Count P95={values['count_p95']:,.8f}, "
        f"Share P95={values['share_p95']:,.8f}"
    )

threshold_table = pd.DataFrame([
    {
        "Mechanism": mechanism,
        "Feature": FEATURES[mechanism],
        "Flow_Lower_P01": behavioral_flow_thresholds[mechanism]["lower_p01"],
        "Flow_Upper_P99": behavioral_flow_thresholds[mechanism]["upper_p99"],
        "Window_Count_P95": behavioral_window_thresholds[mechanism]["count_p95"],
        "Window_Share_P95": behavioral_window_thresholds[mechanism]["share_p95"],
    }
    for mechanism in FEATURES
])

threshold_table.to_csv(
    "/content/csecic2018_frozen_behavioral_thresholds.csv",
    index=False,
)

pd.DataFrame([structural_thresholds]).to_csv(
    "/content/csecic2018_frozen_structural_thresholds.csv",
    index=False,
)

print("\nCreated objects:")
print("  csv_files")
print("  structural_thresholds")
print("  behavioral_flow_thresholds")
print("  behavioral_window_thresholds")

In [ ]:
# ============================================================
# CSE-CIC-IDS2018 — RESUME CALIBRATION FROM SECOND PASS
#
# Run this after the previous cell stopped in the second pass.
# It reuses:
#   CALIBRATION_FILE
#   ZIP_PATH
#   USE_COLUMNS
#   CHUNK_SIZE
#   CAL_START
#   CAL_END
#   FEATURES
#   behavioral_flow_thresholds
#   structural_thresholds
# ============================================================

import gc
import zipfile
import numpy as np
import pandas as pd

required_objects = [
    "CALIBRATION_FILE",
    "ZIP_PATH",
    "USE_COLUMNS",
    "CHUNK_SIZE",
    "CAL_START",
    "CAL_END",
    "FEATURES",
    "behavioral_flow_thresholds",
    "structural_thresholds",
    "norm_text",
    "attempted_mask",
    "ip_hash",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing objects: {missing_objects}\n"
        "The runtime may have restarted. In that case, rerun "
        "the corrected first calibration pass."
    )

print("=" * 110)
print("RESUMING CALIBRATION — WINDOW-LEVEL BEHAVIORAL THRESHOLDS")
print("=" * 110)

# Discard any partial second-pass results created before the error
cal_behavior_parts = []
second_pass_eligible_flows = 0

with zipfile.ZipFile(ZIP_PATH) as archive:

    with archive.open(CALIBRATION_FILE) as stream:

        reader = pd.read_csv(
            stream,
            usecols=USE_COLUMNS,
            chunksize=CHUNK_SIZE,
            low_memory=False,
        )

        for chunk_number, chunk in enumerate(
            reader,
            start=1,
        ):
            timestamp = pd.to_datetime(
                chunk["Timestamp"],
                errors="coerce",
                dayfirst=True,
                format="mixed",
            )

            labels = norm_text(
                chunk["Label"]
            )

            attempted = attempted_mask(
                chunk["Attempted Category"]
            )

            calibration_mask = (
                timestamp.notna()
                & timestamp.ge(CAL_START)
                & timestamp.lt(CAL_END)
                & labels.eq("BENIGN")
                & ~attempted
            )

            if not calibration_mask.any():
                continue

            # Critical correction:
            # reset the index immediately after filtering.
            part = (
                chunk.loc[calibration_mask]
                .copy()
                .reset_index(drop=True)
            )

            part_timestamp = (
                timestamp.loc[calibration_mask]
                .reset_index(drop=True)
            )

            source = (
                part["Src IP"]
                .astype(str)
                .str.strip()
                .reset_index(drop=True)
            )

            # Positional NumPy mask: no Pandas index alignment
            valid_source_mask = (
                source.ne("")
                & source.str.lower().ne("nan")
            ).to_numpy(dtype=bool)

            valid_positions = np.flatnonzero(
                valid_source_mask
            )

            part = (
                part.iloc[valid_positions]
                .reset_index(drop=True)
            )

            part_timestamp = (
                part_timestamp.iloc[valid_positions]
                .reset_index(drop=True)
            )

            source = (
                source.iloc[valid_positions]
                .reset_index(drop=True)
            )

            if not (
                len(part)
                == len(part_timestamp)
                == len(source)
            ):
                raise AssertionError(
                    "Second-pass positional alignment failed."
                )

            second_pass_eligible_flows += len(part)

            output = pd.DataFrame({
                "src_hash": (
                    ip_hash(source)
                    .to_numpy()
                ),

                "window_ns": (
                    part_timestamp
                    .dt.floor("1h")
                    .astype("int64")
                    .to_numpy()
                ),
            })

            aggregation_specification = {}

            for mechanism, feature in FEATURES.items():

                values = pd.to_numeric(
                    part[feature],
                    errors="coerce",
                ).to_numpy(
                    dtype="float64"
                )

                valid_values = np.isfinite(
                    values
                )

                lower_threshold = (
                    behavioral_flow_thresholds[
                        mechanism
                    ]["lower_p01"]
                )

                upper_threshold = (
                    behavioral_flow_thresholds[
                        mechanism
                    ]["upper_p99"]
                )

                anomalous_values = (
                    valid_values
                    & (
                        (values < lower_threshold)
                        | (values > upper_threshold)
                    )
                )

                mechanism_lower = (
                    mechanism.lower()
                )

                valid_column = (
                    f"{mechanism_lower}_valid"
                )

                outlier_column = (
                    f"{mechanism_lower}_outlier"
                )

                output[valid_column] = (
                    valid_values.astype(
                        "int8"
                    )
                )

                output[outlier_column] = (
                    anomalous_values.astype(
                        "int8"
                    )
                )

                aggregation_specification[
                    f"{mechanism_lower}_valid_count"
                ] = (
                    valid_column,
                    "sum",
                )

                aggregation_specification[
                    f"{mechanism_lower}_outlier_count"
                ] = (
                    outlier_column,
                    "sum",
                )

            chunk_behavior = (
                output
                .groupby(
                    [
                        "src_hash",
                        "window_ns",
                    ],
                    observed=True,
                )
                .agg(
                    **aggregation_specification
                )
                .reset_index()
            )

            cal_behavior_parts.append(
                chunk_behavior
            )

            print(
                f"Chunk {chunk_number}: "
                f"eligible flows="
                f"{second_pass_eligible_flows:,}"
            )

# ============================================================
# CONSOLIDATE CHUNKS
# ============================================================

if not cal_behavior_parts:
    raise RuntimeError(
        "No behavioral calibration windows were generated."
    )

cal_behavior = (
    pd.concat(
        cal_behavior_parts,
        ignore_index=True,
    )
    .groupby(
        [
            "src_hash",
            "window_ns",
        ],
        observed=True,
    )
    .sum(
        numeric_only=True
    )
    .reset_index()
)

for mechanism in FEATURES:

    mechanism_lower = (
        mechanism.lower()
    )

    valid_count_column = (
        f"{mechanism_lower}_valid_count"
    )

    outlier_count_column = (
        f"{mechanism_lower}_outlier_count"
    )

    outlier_share_column = (
        f"{mechanism_lower}_outlier_share"
    )

    cal_behavior[
        outlier_share_column
    ] = (
        cal_behavior[
            outlier_count_column
        ]
        /
        cal_behavior[
            valid_count_column
        ].replace(
            0,
            np.nan,
        )
    ).fillna(0)

# ============================================================
# FROZEN WINDOW-LEVEL P95 THRESHOLDS
# ============================================================

behavioral_window_thresholds = {}

for mechanism in FEATURES:

    mechanism_lower = (
        mechanism.lower()
    )

    count_column = (
        f"{mechanism_lower}_outlier_count"
    )

    share_column = (
        f"{mechanism_lower}_outlier_share"
    )

    behavioral_window_thresholds[
        mechanism
    ] = {
        "count_p95": float(
            cal_behavior[
                count_column
            ].quantile(0.95)
        ),

        "share_p95": float(
            cal_behavior[
                share_column
            ].quantile(0.95)
        ),
    }

# ============================================================
# INTEGRITY CHECKS
# ============================================================

EXPECTED_CALIBRATION_FLOWS = 3_507_013

if (
    second_pass_eligible_flows
    != EXPECTED_CALIBRATION_FLOWS
):
    raise AssertionError(
        "Calibration flow-count mismatch: "
        f"expected {EXPECTED_CALIBRATION_FLOWS:,}, "
        f"obtained {second_pass_eligible_flows:,}."
    )

if cal_behavior.duplicated(
    [
        "src_hash",
        "window_ns",
    ]
).any():
    raise AssertionError(
        "Duplicate source-window rows remain "
        "after consolidation."
    )

print("\nSecond-pass positional consistency: PASSED")

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 110)
print("CALIBRATION SUMMARY")
print("=" * 110)

print(
    f"Eligible calibration flows: "
    f"{second_pass_eligible_flows:,}"
)

print(
    f"Behavioral source-windows:  "
    f"{len(cal_behavior):,}"
)

print("\nStructural thresholds:")

for threshold_name, threshold_value in (
    structural_thresholds.items()
):
    print(
        f"  {threshold_name}: "
        f"{threshold_value:,.8f}"
    )

print("\nBehavioral flow thresholds:")

for mechanism, values in (
    behavioral_flow_thresholds.items()
):
    print(
        f"  {mechanism}: "
        f"P01={values['lower_p01']:,.8f}, "
        f"P99={values['upper_p99']:,.8f}, "
        f"N={values['valid_flows']:,}"
    )

print("\nBehavioral window thresholds:")

for mechanism, values in (
    behavioral_window_thresholds.items()
):
    print(
        f"  {mechanism}: "
        f"Count P95="
        f"{values['count_p95']:,.8f}, "
        f"Share P95="
        f"{values['share_p95']:,.8f}"
    )

# ============================================================
# SAVE THRESHOLDS
# ============================================================

behavioral_threshold_table = pd.DataFrame(
    [
        {
            "Mechanism": mechanism,
            "Feature": FEATURES[
                mechanism
            ],

            "Flow_Lower_P01": (
                behavioral_flow_thresholds[
                    mechanism
                ]["lower_p01"]
            ),

            "Flow_Upper_P99": (
                behavioral_flow_thresholds[
                    mechanism
                ]["upper_p99"]
            ),

            "Window_Count_P95": (
                behavioral_window_thresholds[
                    mechanism
                ]["count_p95"]
            ),

            "Window_Share_P95": (
                behavioral_window_thresholds[
                    mechanism
                ]["share_p95"]
            ),
        }
        for mechanism in FEATURES
    ]
)

behavioral_threshold_table.to_csv(
    "/content/"
    "csecic2018_frozen_behavioral_thresholds.csv",
    index=False,
)

pd.DataFrame(
    [structural_thresholds]
).to_csv(
    "/content/"
    "csecic2018_frozen_structural_thresholds.csv",
    index=False,
)

print("\nObjects ready:")
print("  structural_thresholds")
print("  behavioral_flow_thresholds")
print("  behavioral_window_thresholds")
print("  cal_behavior")

print("\nSaved:")
print(
    "/content/"
    "csecic2018_frozen_behavioral_thresholds.csv"
)
print(
    "/content/"
    "csecic2018_frozen_structural_thresholds.csv"
)

del cal_behavior_parts
gc.collect()

In [ ]:
# ============================================================
# CSE-CIC-IDS2018 — FROZEN STREAMING EVALUATION
# Corrected disk-backed implementation using DuckDB
# ============================================================

import os
import re
import gc
import sys
import shutil
import zipfile
import subprocess

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "duckdb"]
    )
    import duckdb


print("=" * 120)
print("CSE-CIC-IDS2018 — FROZEN STREAMING EVALUATION")
print("=" * 120)

# ============================================================
# 1. VERIFY REQUIRED OBJECTS
# ============================================================

required_objects = [
    "ZIP_PATH",
    "EVAL_START",
    "FEATURES",
    "structural_thresholds",
    "behavioral_flow_thresholds",
    "behavioral_window_thresholds",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required objects: {missing_objects}\n"
        "The Colab runtime may have restarted."
    )

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(ZIP_PATH)

# ============================================================
# 2. PATHS AND CLEAN START
# ============================================================

DB_PATH = "/content/csecic2018_frozen_evaluation.duckdb"
TEMP_DIRECTORY = "/content/csecic2018_duckdb_temp"
EXTRACT_DIRECTORY = "/content/csecic2018_current_csv"

for path in [
    DB_PATH,
    DB_PATH + ".wal",
]:
    if os.path.exists(path):
        os.remove(path)

for directory in [
    TEMP_DIRECTORY,
    EXTRACT_DIRECTORY,
]:
    if os.path.exists(directory):
        shutil.rmtree(directory)

    os.makedirs(
        directory,
        exist_ok=True,
    )

total_disk, used_disk, free_disk = shutil.disk_usage(
    "/content"
)

print(
    f"Free disk before evaluation: "
    f"{free_disk / 1024**3:,.2f} GB"
)

if free_disk / 1024**3 < 20:
    raise RuntimeError(
        "Less than 20 GB of local disk is available. "
        "Restart with a runtime providing more storage."
    )

# ============================================================
# 3. FILE ORDER
# ============================================================

def filename_date(filename):
    match = re.search(
        r"(\d{2}-\d{2}-\d{4})",
        os.path.basename(filename),
    )

    if not match:
        return pd.NaT

    return pd.to_datetime(
        match.group(1),
        format="%d-%m-%Y",
        errors="coerce",
    )


with zipfile.ZipFile(ZIP_PATH) as archive:

    csv_files = sorted(
        [
            info.filename
            for info in archive.infolist()
            if (
                not info.is_dir()
                and info.filename.lower().endswith(".csv")
            )
        ],
        key=filename_date,
    )

if not csv_files:
    raise RuntimeError(
        "No CSV files were found in the archive."
    )

print(f"Evaluation files: {len(csv_files):,}")

for filename in csv_files:
    print(" ", filename)

# ============================================================
# 4. DUCKDB DATABASE
# ============================================================

con = duckdb.connect(
    DB_PATH
)

con.execute(
    f"PRAGMA temp_directory='{TEMP_DIRECTORY}'"
)

con.execute(
    "PRAGMA preserve_insertion_order=false"
)

# Partial source-window statistics
con.execute("""
CREATE TABLE window_parts (
    src VARCHAR,
    window_start TIMESTAMP,
    flow_count BIGINT,
    first_time TIMESTAMP,
    last_time TIMESTAMP,
    primary_attack_count BIGINT,
    sensitivity_attack_count BIGINT,
    packet_valid_count BIGINT,
    packet_outlier_count BIGINT,
    temporal_valid_count BIGINT,
    temporal_outlier_count BIGINT,
    directional_valid_count BIGINT,
    directional_outlier_count BIGINT
)
""")

# Exact source-window-destination edges
con.execute("""
CREATE TABLE edge_parts (
    src VARCHAR,
    window_start TIMESTAMP,
    dst VARCHAR
)
""")

# Effective attack labels for post-hoc family attribution
con.execute("""
CREATE TABLE label_parts (
    src VARCHAR,
    window_start TIMESTAMP,
    attack_label VARCHAR,
    attack_label_flows BIGINT
)
""")

# ============================================================
# 5. LOCKED NUMERIC THRESHOLDS
# ============================================================

packet_lower = float(
    behavioral_flow_thresholds[
        "PACKET"
    ]["lower_p01"]
)

packet_upper = float(
    behavioral_flow_thresholds[
        "PACKET"
    ]["upper_p99"]
)

temporal_lower = float(
    behavioral_flow_thresholds[
        "TEMPORAL"
    ]["lower_p01"]
)

temporal_upper = float(
    behavioral_flow_thresholds[
        "TEMPORAL"
    ]["upper_p99"]
)

directional_lower = float(
    behavioral_flow_thresholds[
        "DIRECTIONAL"
    ]["lower_p01"]
)

directional_upper = float(
    behavioral_flow_thresholds[
        "DIRECTIONAL"
    ]["upper_p99"]
)

evaluation_start_sql = pd.Timestamp(
    EVAL_START
).strftime(
    "%Y-%m-%d %H:%M:%S"
)

# ============================================================
# 6. PROCESS ONE CSV AT A TIME
# ============================================================

total_evaluation_flows = 0
processed_files = []

with zipfile.ZipFile(ZIP_PATH) as archive:

    for file_number, member_name in enumerate(
        csv_files,
        start=1,
    ):
        print(
            "\n"
            + "=" * 100
        )
        print(
            f"[{file_number:02d}/{len(csv_files):02d}] "
            f"{member_name}"
        )

        extracted_path = archive.extract(
            member_name,
            path=EXTRACT_DIRECTORY,
        )

        sql_path = extracted_path.replace(
            "'",
            "''",
        )

        # ----------------------------------------------------
        # Materialize only required columns for this file
        # ----------------------------------------------------

        con.execute(
            "DROP TABLE IF EXISTS current_eval"
        )

        con.execute(f"""
        CREATE TEMP TABLE current_eval AS
        WITH parsed AS (
            SELECT
                TRY_CAST("Timestamp" AS TIMESTAMP) AS ts,

                TRIM(
                    CAST("Src IP" AS VARCHAR)
                ) AS src,

                TRIM(
                    CAST("Dst IP" AS VARCHAR)
                ) AS dst,

                TRY_CAST(
                    "Flow IAT Max" AS DOUBLE
                ) AS temporal_value,

                TRY_CAST(
                    "Bwd Segment Size Avg" AS DOUBLE
                ) AS packet_value,

                TRY_CAST(
                    "Down/Up Ratio" AS DOUBLE
                ) AS directional_value,

                UPPER(
                    TRIM(
                        CAST("Label" AS VARCHAR)
                    )
                ) AS label_value,

                TRY_CAST(
                    "Attempted Category" AS DOUBLE
                ) AS attempted_category

            FROM read_csv_auto(
                '{sql_path}',
                header=true,
                all_varchar=true
            )
        )

        SELECT *
        FROM parsed

        WHERE
            ts >= TIMESTAMP '{evaluation_start_sql}'

            AND src IS NOT NULL
            AND dst IS NOT NULL

            AND src <> ''
            AND dst <> ''

            AND LOWER(src) <> 'nan'
            AND LOWER(dst) <> 'nan'
        """)

        current_count = int(
            con.execute(
                "SELECT COUNT(*) FROM current_eval"
            ).fetchone()[0]
        )

        total_evaluation_flows += current_count

        print(
            f"  Evaluation flows in file: "
            f"{current_count:,}"
        )

        # ----------------------------------------------------
        # Source-window aggregate parts
        # ----------------------------------------------------

        con.execute(f"""
        INSERT INTO window_parts

        SELECT
            src,

            DATE_TRUNC(
                'hour',
                ts
            ) AS window_start,

            COUNT(*) AS flow_count,

            MIN(ts) AS first_time,

            MAX(ts) AS last_time,

            SUM(
                CASE
                    WHEN
                        label_value <> 'BENIGN'
                        AND COALESCE(
                            attempted_category,
                            -1
                        ) = -1
                    THEN 1
                    ELSE 0
                END
            ) AS primary_attack_count,

            SUM(
                CASE
                    WHEN label_value <> 'BENIGN'
                    THEN 1
                    ELSE 0
                END
            ) AS sensitivity_attack_count,

            SUM(
                CASE
                    WHEN
                        packet_value IS NOT NULL
                        AND ISFINITE(packet_value)
                    THEN 1
                    ELSE 0
                END
            ) AS packet_valid_count,

            SUM(
                CASE
                    WHEN
                        packet_value IS NOT NULL
                        AND ISFINITE(packet_value)
                        AND (
                            packet_value < {packet_lower!r}
                            OR packet_value > {packet_upper!r}
                        )
                    THEN 1
                    ELSE 0
                END
            ) AS packet_outlier_count,

            SUM(
                CASE
                    WHEN
                        temporal_value IS NOT NULL
                        AND ISFINITE(temporal_value)
                    THEN 1
                    ELSE 0
                END
            ) AS temporal_valid_count,

            SUM(
                CASE
                    WHEN
                        temporal_value IS NOT NULL
                        AND ISFINITE(temporal_value)
                        AND (
                            temporal_value < {temporal_lower!r}
                            OR temporal_value > {temporal_upper!r}
                        )
                    THEN 1
                    ELSE 0
                END
            ) AS temporal_outlier_count,

            SUM(
                CASE
                    WHEN
                        directional_value IS NOT NULL
                        AND ISFINITE(directional_value)
                    THEN 1
                    ELSE 0
                END
            ) AS directional_valid_count,

            SUM(
                CASE
                    WHEN
                        directional_value IS NOT NULL
                        AND ISFINITE(directional_value)
                        AND (
                            directional_value < {directional_lower!r}
                            OR directional_value > {directional_upper!r}
                        )
                    THEN 1
                    ELSE 0
                END
            ) AS directional_outlier_count

        FROM current_eval

        GROUP BY
            src,
            DATE_TRUNC('hour', ts)
        """)

        # ----------------------------------------------------
        # Exact unique destinations
        # ----------------------------------------------------

        con.execute("""
        INSERT INTO edge_parts

        SELECT DISTINCT
            src,
            DATE_TRUNC(
                'hour',
                ts
            ) AS window_start,
            dst

        FROM current_eval
        """)

        # ----------------------------------------------------
        # Effective primary attack labels
        # ----------------------------------------------------

        con.execute("""
        INSERT INTO label_parts

        SELECT
            src,

            DATE_TRUNC(
                'hour',
                ts
            ) AS window_start,

            label_value AS attack_label,

            COUNT(*) AS attack_label_flows

        FROM current_eval

        WHERE
            label_value <> 'BENIGN'

            AND COALESCE(
                attempted_category,
                -1
            ) = -1

        GROUP BY
            src,
            DATE_TRUNC('hour', ts),
            label_value
        """)

        con.execute(
            "DROP TABLE current_eval"
        )

        con.execute(
            "CHECKPOINT"
        )

        processed_files.append(
            member_name
        )

        # Delete extracted file immediately
        try:
            os.remove(
                extracted_path
            )
        except OSError:
            pass

        # Remove now-empty nested folders
        for root, directories, files in os.walk(
            EXTRACT_DIRECTORY,
            topdown=False,
        ):
            if (
                not directories
                and not files
                and root != EXTRACT_DIRECTORY
            ):
                try:
                    os.rmdir(root)
                except OSError:
                    pass

        free_now = shutil.disk_usage(
            "/content"
        ).free

        print(
            f"  Cumulative evaluation flows: "
            f"{total_evaluation_flows:,}"
        )

        print(
            f"  Remaining disk: "
            f"{free_now / 1024**3:,.2f} GB"
        )

# ============================================================
# 7. CONSOLIDATE SOURCE-WINDOW STATISTICS
# ============================================================

print("\n" + "=" * 120)
print("CONSOLIDATING SOURCE-WINDOW TABLES")
print("=" * 120)

con.execute("""
CREATE TABLE final_window_statistics AS

SELECT
    src,
    window_start,

    SUM(flow_count) AS flow_count,

    MIN(first_time) AS first_time,

    MAX(last_time) AS last_time,

    SUM(
        primary_attack_count
    ) AS primary_attack_count,

    SUM(
        sensitivity_attack_count
    ) AS sensitivity_attack_count,

    SUM(
        packet_valid_count
    ) AS packet_valid_count,

    SUM(
        packet_outlier_count
    ) AS packet_outlier_count,

    SUM(
        temporal_valid_count
    ) AS temporal_valid_count,

    SUM(
        temporal_outlier_count
    ) AS temporal_outlier_count,

    SUM(
        directional_valid_count
    ) AS directional_valid_count,

    SUM(
        directional_outlier_count
    ) AS directional_outlier_count

FROM window_parts

GROUP BY
    src,
    window_start
""")

print("Calculating exact unique destinations...")

con.execute("""
CREATE TABLE final_unique_destinations AS

SELECT
    src,
    window_start,
    COUNT(*) AS unique_dst_ip

FROM (
    SELECT DISTINCT
        src,
        window_start,
        dst

    FROM edge_parts
)

GROUP BY
    src,
    window_start
""")

print("Calculating dominant effective attack labels...")

con.execute("""
CREATE TABLE final_dominant_labels AS

WITH combined_labels AS (
    SELECT
        src,
        window_start,
        attack_label,
        SUM(
            attack_label_flows
        ) AS attack_label_flows

    FROM label_parts

    GROUP BY
        src,
        window_start,
        attack_label
),

ranked_labels AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY
                src,
                window_start

            ORDER BY
                attack_label_flows DESC,
                attack_label ASC
        ) AS label_rank

    FROM combined_labels
)

SELECT
    src,
    window_start,
    attack_label AS dominant_attack_label,
    attack_label_flows

FROM ranked_labels

WHERE label_rank = 1
""")

# ============================================================
# 8. FETCH FINAL SOURCE-WINDOW TABLE
# ============================================================

csec_frozen = con.execute("""
SELECT
    w.*,
    d.unique_dst_ip,
    l.dominant_attack_label,
    l.attack_label_flows

FROM final_window_statistics AS w

LEFT JOIN final_unique_destinations AS d
    USING (
        src,
        window_start
    )

LEFT JOIN final_dominant_labels AS l
    USING (
        src,
        window_start
    )

ORDER BY
    window_start,
    src
""").fetchdf()

csec_frozen = (
    csec_frozen
    .rename(
        columns={
            "src": "_src",
            "window_start": "_window_start",
        }
    )
    .reset_index(drop=True)
)

# ============================================================
# 9. DERIVE STRUCTURAL AND BEHAVIORAL OPERATORS
# ============================================================

csec_frozen[
    "activity_duration_seconds"
] = (
    pd.to_datetime(
        csec_frozen["last_time"]
    )
    -
    pd.to_datetime(
        csec_frozen["first_time"]
    )
).dt.total_seconds()

csec_frozen[
    "active_span_flow_rate"
] = (
    csec_frozen["flow_count"]
    /
    csec_frozen[
        "activity_duration_seconds"
    ].clip(lower=1.0)
)

csec_frozen[
    "destination_repetition"
] = (
    csec_frozen["flow_count"]
    /
    csec_frozen[
        "unique_dst_ip"
    ].replace(
        0,
        np.nan,
    )
).fillna(0)

csec_frozen[
    "primary_attack_share"
] = (
    csec_frozen[
        "primary_attack_count"
    ]
    /
    csec_frozen[
        "flow_count"
    ].replace(
        0,
        np.nan,
    )
).fillna(0)

csec_frozen[
    "sensitivity_attack_share"
] = (
    csec_frozen[
        "sensitivity_attack_count"
    ]
    /
    csec_frozen[
        "flow_count"
    ].replace(
        0,
        np.nan,
    )
).fillna(0)

for mechanism in [
    "packet",
    "temporal",
    "directional",
]:
    csec_frozen[
        f"{mechanism}_outlier_share"
    ] = (
        csec_frozen[
            f"{mechanism}_outlier_count"
        ]
        /
        csec_frozen[
            f"{mechanism}_valid_count"
        ].replace(
            0,
            np.nan,
        )
    ).fillna(0)

# Structural operators
csec_frozen["S_BV"] = (
    (
        csec_frozen[
            "active_span_flow_rate"
        ]
        >= structural_thresholds[
            "BV_RATE_P99"
        ]
    )
    &
    (
        csec_frozen[
            "flow_count"
        ]
        >= structural_thresholds[
            "BV_COUNT_P99"
        ]
    )
)

csec_frozen["S_REP"] = (
    (
        csec_frozen[
            "destination_repetition"
        ]
        >= structural_thresholds[
            "REP_RATIO_P99"
        ]
    )
    &
    (
        csec_frozen[
            "flow_count"
        ]
        >= structural_thresholds[
            "REP_COUNT_P50"
        ]
    )
)

csec_frozen["S_T"] = (
    csec_frozen["S_BV"]
    |
    csec_frozen["S_REP"]
)

# Behavioral threshold helper
def apply_threshold(
    series,
    threshold,
):
    if threshold <= 0:
        return series.gt(0)

    return series.ge(
        threshold
    )


# Packet: JOINT P95
packet_count_condition = apply_threshold(
    csec_frozen[
        "packet_outlier_count"
    ],
    behavioral_window_thresholds[
        "PACKET"
    ]["count_p95"],
)

packet_share_condition = apply_threshold(
    csec_frozen[
        "packet_outlier_share"
    ],
    behavioral_window_thresholds[
        "PACKET"
    ]["share_p95"],
)

csec_frozen["B_PACKET"] = (
    packet_count_condition
    &
    packet_share_condition
)

# Temporal: COUNT P95
csec_frozen["B_TEMPORAL"] = (
    apply_threshold(
        csec_frozen[
            "temporal_outlier_count"
        ],
        behavioral_window_thresholds[
            "TEMPORAL"
        ]["count_p95"],
    )
)

# Directional: COUNT P95
csec_frozen["B_DIRECTIONAL"] = (
    apply_threshold(
        csec_frozen[
            "directional_outlier_count"
        ],
        behavioral_window_thresholds[
            "DIRECTIONAL"
        ]["count_p95"],
    )
)

csec_frozen["B_MULTI"] = (
    csec_frozen["B_PACKET"]
    |
    csec_frozen["B_TEMPORAL"]
    |
    csec_frozen["B_DIRECTIONAL"]
)

csec_frozen["H_MULTI"] = (
    csec_frozen["S_T"]
    &
    csec_frozen["B_MULTI"]
)

csec_frozen[
    "behavioral_mechanism_count"
] = (
    csec_frozen[
        [
            "B_PACKET",
            "B_TEMPORAL",
            "B_DIRECTIONAL",
        ]
    ]
    .astype(int)
    .sum(axis=1)
)

csec_frozen["dataset_day"] = (
    csec_frozen[
        "_window_start"
    ].dt.day_name()
)

# ============================================================
# 10. INTEGRITY CHECKS
# ============================================================

if csec_frozen.duplicated(
    [
        "_src",
        "_window_start",
    ]
).any():
    raise AssertionError(
        "Duplicate source-window rows remain."
    )

if int(
    csec_frozen[
        "flow_count"
    ].sum()
) != total_evaluation_flows:
    raise AssertionError(
        "Flow-count integrity failure: "
        f"aggregated={int(csec_frozen['flow_count'].sum()):,}, "
        f"streamed={total_evaluation_flows:,}"
    )

print("\nEvaluation aggregation integrity: PASSED")

print("\n" + "=" * 120)
print("STREAMING AGGREGATION COMPLETE")
print("=" * 120)

print(
    f"Processed files:           "
    f"{len(processed_files):,}"
)

print(
    f"Evaluation flows:          "
    f"{total_evaluation_flows:,}"
)

print(
    f"Evaluation source-windows: "
    f"{len(csec_frozen):,}"
)

print(
    f"Primary attack windows:    "
    f"{(csec_frozen['primary_attack_count'] > 0).sum():,}"
)

print(
    f"Primary dominant windows:  "
    f"{(csec_frozen['primary_attack_share'] >= 0.50).sum():,}"
)

print(
    f"S_BV alerts:               "
    f"{csec_frozen['S_BV'].sum():,}"
)

print(
    f"S_REP alerts:              "
    f"{csec_frozen['S_REP'].sum():,}"
)

print(
    f"S_T alerts:                "
    f"{csec_frozen['S_T'].sum():,}"
)

print(
    f"B_MULTI alerts:            "
    f"{csec_frozen['B_MULTI'].sum():,}"
)

print(
    f"H_MULTI alerts:            "
    f"{csec_frozen['H_MULTI'].sum():,}"
)

# Save compact source-window output
csec_frozen.to_parquet(
    "/content/csecic2018_frozen_source_windows.parquet",
    index=False,
)

print("\nCreated object: csec_frozen")
print("DuckDB audit database:", DB_PATH)
print(
    "Saved: "
    "/content/csecic2018_frozen_source_windows.parquet"
)

con.close()
gc.collect()

In [ ]:
# ============================================================
# CSE-CIC-IDS2018 — INDEPENDENT FROZEN VALIDATION METRICS
# ============================================================

import math
import numpy as np
import pandas as pd

try:
    from scipy.stats import hypergeom
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False


def bool_array(values):
    return np.asarray(
        values,
        dtype=bool,
    )


def evaluate_model(
    prediction,
    truth,
    frame,
):
    prediction = bool_array(
        prediction
    )

    truth = bool_array(
        truth
    )

    if (
        len(prediction) != len(truth)
        or len(prediction) != len(frame)
    ):
        raise ValueError(
            "Prediction, truth and frame lengths differ."
        )

    tp_mask = (
        prediction
        &
        truth
    )

    fp_mask = (
        prediction
        &
        ~truth
    )

    fn_mask = (
        ~prediction
        &
        truth
    )

    tn_mask = (
        ~prediction
        &
        ~truth
    )

    tp = int(tp_mask.sum())
    fp = int(fp_mask.sum())
    fn = int(fn_mask.sum())
    tn = int(tn_mask.sum())

    flagged = int(
        prediction.sum()
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    base_rate = float(
        truth.mean()
    )

    lift = (
        precision / base_rate
        if base_rate > 0
        else np.nan
    )

    return {
        "Flagged": flagged,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "Base_Rate": base_rate,
        "Lift": lift,

        "Unique_Flagged_Sources": (
            frame.loc[
                prediction,
                "_src",
            ].nunique()
        ),

        "Unique_TP_Sources": (
            frame.loc[
                tp_mask,
                "_src",
            ].nunique()
        ),

        "Mean_Attack_Share_Flagged": (
            frame.loc[
                prediction,
                "primary_attack_share",
            ].mean()
            if flagged > 0
            else np.nan
        ),

        "Median_Attack_Share_Flagged": (
            frame.loc[
                prediction,
                "primary_attack_share",
            ].median()
            if flagged > 0
            else np.nan
        ),
    }


def calculate_rci(
    structural_prediction,
    behavioral_prediction,
):
    S = bool_array(
        structural_prediction
    )

    B = bool_array(
        behavioral_prediction
    )

    n = len(S)

    s_size = int(S.sum())
    b_size = int(B.sum())

    intersection = int(
        (
            S
            &
            B
        ).sum()
    )

    union = int(
        (
            S
            |
            B
        ).sum()
    )

    observed_rci = (
        intersection / union
        if union > 0
        else 0.0
    )

    s_rate = (
        s_size / n
        if n > 0
        else 0.0
    )

    b_rate = (
        b_size / n
        if n > 0
        else 0.0
    )

    expected_intersection = (
        s_rate * b_rate
    )

    expected_union = (
        s_rate
        +
        b_rate
        -
        expected_intersection
    )

    expected_rci = (
        expected_intersection
        /
        expected_union
        if expected_union > 0
        else 0.0
    )

    p_value = np.nan
    log10_p = np.nan

    if SCIPY_AVAILABLE:
        p_value = float(
            hypergeom.sf(
                intersection - 1,
                n,
                s_size,
                b_size,
            )
        )

        log_sf = float(
            hypergeom.logsf(
                intersection - 1,
                n,
                s_size,
                b_size,
            )
        )

        if np.isfinite(log_sf):
            log10_p = (
                log_sf
                /
                math.log(10)
            )

    return {
        "Evaluation_Units": n,
        "S_T_Size": s_size,
        "B_MULTI_Size": b_size,
        "Intersection_Size": intersection,
        "Union_Size": union,
        "RCI_Jaccard": observed_rci,
        "Expected_RCI": expected_rci,

        "RCI_Lift": (
            observed_rci / expected_rci
            if expected_rci > 0
            else np.nan
        ),

        "Coverage_S_to_B": (
            intersection / s_size
            if s_size > 0
            else 0.0
        ),

        "Coverage_B_to_S": (
            intersection / b_size
            if b_size > 0
            else 0.0
        ),

        "Hypergeometric_P": p_value,
        "Hypergeometric_Log10_P": log10_p,

        "S_Subset_B": bool(
            (
                S
                &
                ~B
            ).sum() == 0
        ),

        "B_Subset_S": bool(
            (
                B
                &
                ~S
            ).sum() == 0
        ),
    }


models = {
    "S_BV": csec_frozen["S_BV"],
    "S_REP": csec_frozen["S_REP"],
    "S_T": csec_frozen["S_T"],
    "B_PACKET": csec_frozen["B_PACKET"],
    "B_TEMPORAL": csec_frozen["B_TEMPORAL"],
    "B_DIRECTIONAL": csec_frozen["B_DIRECTIONAL"],
    "B_MULTI": csec_frozen["B_MULTI"],
    "H_MULTI": csec_frozen["H_MULTI"],
}

truths = {
    "Primary_Any_Attack": (
        csec_frozen[
            "primary_attack_count"
        ] > 0
    ),

    "Primary_AttackShare_GE_50pct": (
        csec_frozen[
            "primary_attack_share"
        ] >= 0.50
    ),

    "Primary_AttackShare_GE_99pct": (
        csec_frozen[
            "primary_attack_share"
        ] >= 0.99
    ),

    "Sensitivity_Any_Attack": (
        csec_frozen[
            "sensitivity_attack_count"
        ] > 0
    ),

    "Sensitivity_AttackShare_GE_50pct": (
        csec_frozen[
            "sensitivity_attack_share"
        ] >= 0.50
    ),
}

metric_rows = []

for truth_name, truth in truths.items():

    for model_name, prediction in models.items():

        metric_rows.append({
            "Model": model_name,
            "Truth_Definition": truth_name,

            **evaluate_model(
                prediction,
                truth,
                csec_frozen,
            ),
        })

csec_frozen_metrics = pd.DataFrame(
    metric_rows
)

csec_frozen_rci = pd.DataFrame(
    [
        calculate_rci(
            csec_frozen["S_T"],
            csec_frozen["B_MULTI"],
        )
    ]
)

# ============================================================
# FAMILY COVERAGE
# ============================================================

family_rows = []

primary_dominant = (
    csec_frozen[
        csec_frozen[
            "primary_attack_share"
        ] >= 0.50
    ]
    .copy()
)

for model_name in [
    "S_T",
    "B_MULTI",
    "H_MULTI",
]:
    primary_dominant[
        "_detected"
    ] = (
        primary_dominant[
            model_name
        ]
        .astype(bool)
    )

    for family, group in (
        primary_dominant
        .dropna(
            subset=[
                "dominant_attack_label"
            ]
        )
        .groupby(
            "dominant_attack_label",
            observed=True,
        )
    ):
        detected = group[
            group["_detected"]
        ]

        family_rows.append({
            "Model": model_name,
            "Attack_Family": family,
            "Attack_Dominant_Windows": (
                len(group)
            ),
            "Detected_Windows": (
                len(detected)
            ),
            "Family_Recall": (
                len(detected) / len(group)
                if len(group) > 0
                else np.nan
            ),
            "Unique_Attack_Sources": (
                group["_src"].nunique()
            ),
            "Detected_Sources": (
                detected["_src"].nunique()
            ),
            "Total_Primary_Attack_Flows": int(
                group[
                    "primary_attack_count"
                ].sum()
            ),
        })

csec_frozen_family = pd.DataFrame(
    family_rows
)

# ============================================================
# STRUCTURAL ALERT ATTRIBUTION
# ============================================================

csec_structural_alerts = (
    csec_frozen[
        csec_frozen["S_T"]
    ]
    .copy()
    .reset_index(drop=True)
)

csec_structural_alerts[
    "primary_attack_dominant"
] = (
    csec_structural_alerts[
        "primary_attack_share"
    ] >= 0.50
)

# ============================================================
# CONSISTENCY CHECKS
# ============================================================

h_primary = (
    csec_frozen_metrics[
        (
            csec_frozen_metrics[
                "Model"
            ].eq("H_MULTI")
        )
        &
        (
            csec_frozen_metrics[
                "Truth_Definition"
            ].eq(
                "Primary_AttackShare_GE_50pct"
            )
        )
    ]
    .iloc[0]
)

direct_h = (
    csec_frozen["H_MULTI"]
    .to_numpy(
        dtype=bool
    )
)

direct_truth = (
    csec_frozen[
        "primary_attack_share"
    ]
    .ge(0.50)
    .to_numpy(
        dtype=bool
    )
)

assert int(
    (
        direct_h
        &
        direct_truth
    ).sum()
) == int(
    h_primary["TP"]
)

assert int(
    (
        direct_h
        &
        ~direct_truth
    ).sum()
) == int(
    h_primary["FP"]
)

print("Consistency checks: PASSED")

# ============================================================
# DISPLAY PRIMARY RESULTS
# ============================================================

print("\n" + "=" * 120)
print(
    "INDEPENDENT FROZEN VALIDATION "
    "— PRIMARY ATTACK SHARE >= 50%"
)
print("=" * 120)

primary_results = (
    csec_frozen_metrics[
        csec_frozen_metrics[
            "Truth_Definition"
        ].eq(
            "Primary_AttackShare_GE_50pct"
        )
    ]
    .sort_values(
        [
            "F1",
            "Precision",
            "Recall",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
)

print(
    primary_results
    .round(8)
    .to_string(index=False)
)

print("\n" + "=" * 120)
print(
    "INDEPENDENT FROZEN VALIDATION "
    "— SENSITIVITY ATTACK SHARE >= 50%"
)
print("=" * 120)

sensitivity_results = (
    csec_frozen_metrics[
        csec_frozen_metrics[
            "Truth_Definition"
        ].eq(
            "Sensitivity_AttackShare_GE_50pct"
        )
    ]
    .sort_values(
        [
            "F1",
            "Precision",
            "Recall",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
)

print(
    sensitivity_results
    .round(8)
    .to_string(index=False)
)

print("\n" + "=" * 120)
print("INDEPENDENT TEMPORAL RCI")
print("=" * 120)

print(
    csec_frozen_rci
    .round(12)
    .to_string(index=False)
)

print("\n" + "=" * 120)
print("STRUCTURAL ALERT ATTRIBUTION")
print("=" * 120)

display_columns = [
    "_src",
    "_window_start",
    "dominant_attack_label",
    "flow_count",
    "unique_dst_ip",
    "active_span_flow_rate",
    "destination_repetition",
    "primary_attack_count",
    "primary_attack_share",
    "sensitivity_attack_count",
    "sensitivity_attack_share",
    "S_BV",
    "S_REP",
    "S_T",
    "B_PACKET",
    "B_TEMPORAL",
    "B_DIRECTIONAL",
    "behavioral_mechanism_count",
    "B_MULTI",
    "H_MULTI",
    "primary_attack_dominant",
]

print(
    csec_structural_alerts[
        display_columns
    ]
    .sort_values(
        [
            "primary_attack_dominant",
            "primary_attack_share",
            "primary_attack_count",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .to_string(
        index=False,
        formatters={
            "primary_attack_share": (
                "{:.2%}".format
            ),
            "sensitivity_attack_share": (
                "{:.2%}".format
            ),
        },
    )
)

print("\n" + "=" * 120)
print("PRIMARY FAMILY COVERAGE — H_MULTI")
print("=" * 120)

print(
    csec_frozen_family[
        csec_frozen_family[
            "Model"
        ].eq("H_MULTI")
    ]
    .sort_values(
        [
            "Detected_Windows",
            "Family_Recall",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .to_string(
        index=False,
        formatters={
            "Family_Recall": (
                "{:.2%}".format
            ),
        },
    )
)

# ============================================================
# SAVE
# ============================================================

csec_frozen_metrics.to_csv(
    "/content/csecic2018_frozen_metrics.csv",
    index=False,
)

csec_frozen_rci.to_csv(
    "/content/csecic2018_frozen_RCI.csv",
    index=False,
)

csec_frozen_family.to_csv(
    "/content/csecic2018_frozen_family_coverage.csv",
    index=False,
)

csec_structural_alerts.to_csv(
    "/content/csecic2018_frozen_structural_alerts.csv",
    index=False,
)

print("\nSaved:")
print("/content/csecic2018_frozen_metrics.csv")
print("/content/csecic2018_frozen_RCI.csv")
print("/content/csecic2018_frozen_family_coverage.csv")
print("/content/csecic2018_frozen_structural_alerts.csv")

Consistency checks: PASSED

INDEPENDENT FROZEN VALIDATION — PRIMARY ATTACK SHARE >= 50%
        Model             Truth_Definition  Flagged  TP    FP  FN     TN  Precision   Recall       F1      FPR  Base_Rate       Lift  Unique_Flagged_Sources  Unique_TP_Sources  Mean_Attack_Share_Flagged  Median_Attack_Share_Flagged
         S_BV Primary_AttackShare_GE_50pct      198  50   148  89 520060   0.252525 0.359712 0.296736 0.000284   0.000267 945.329191                      55                 16                   0.246935                          0.0
        S_REP Primary_AttackShare_GE_50pct     3913 125  3788  14 516420   0.031945 0.899281 0.061698 0.007282   0.000267 119.585471                     434                 26                   0.031522                          0.0
          S_T Primary_AttackShare_GE_50pct     3995 127  3868  12 516340   0.031790 0.913669 0.061442 0.007435   0.000267 119.004995                     460                 26                   0.031252              

## Structural ablation baselines

This section adds post-hoc ablation baselines **without changing the original frozen protocol**.

The purpose is to test whether the conjunctions used by the structural operators add measurable selectivity beyond their single-component threshold rules under the **same source-hour representation, target-environment calibration policy, and primary ground-truth definition**.

Baselines:
- `COUNT_ONLY_P99`: flow count ≥ calibration P99
- `RATE_ONLY_P99`: active-span flow rate ≥ calibration P99
- `REP_ONLY_P99`: destination-repetition ratio ≥ calibration P99

Comparators:
- `S_BV = RATE_ONLY_P99 AND COUNT_ONLY_P99`
- `S_REP = REP_ONLY_P99 AND flow-count P50 guard`
- `S_T = S_BV OR S_REP`

The cell reports the same detection metrics used in the study and saves both the baseline table and direct conjunction-value diagnostics.


# New Section

In [ ]:
# ============================================================
# STRUCTURAL ABLATION BASELINES — IMPROVED CSE-CIC-IDS2018
#
# Purpose:
#   Test whether the SBV/SREP conjunctions add selectivity over
#   their single-component threshold rules under exactly the
#   same source-hour semantics, target-environment calibration,
#   and primary AttackShare >= 50% ground-truth definition used
#   in the paper.
#
# IMPORTANT:
#   This cell does NOT alter the frozen operators or any original
#   result. It adds post-hoc ablation baselines only.
# ============================================================

import numpy as np
import pandas as pd

required_objects = [
    "csec_frozen",
    "structural_thresholds",
    "evaluate_model",
]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the original CSE-CIC-IDS2018 calibration, streaming "
        "evaluation, and metrics cells first. "
        f"Missing objects: {missing}"
    )

ablation_predictions = {
    "COUNT_ONLY_P99": (
        csec_frozen["flow_count"]
        >= structural_thresholds["BV_COUNT_P99"]
    ),
    "RATE_ONLY_P99": (
        csec_frozen["active_span_flow_rate"]
        >= structural_thresholds["BV_RATE_P99"]
    ),
    "REP_ONLY_P99": (
        csec_frozen["destination_repetition"]
        >= structural_thresholds["REP_RATIO_P99"]
    ),
    "S_BV": csec_frozen["S_BV"],
    "S_REP": csec_frozen["S_REP"],
    "S_T": csec_frozen["S_T"],
}

truth_primary = (
    csec_frozen["primary_attack_share"] >= 0.50
)

ablation_rows = []
for model_name, prediction in ablation_predictions.items():
    metrics = evaluate_model(
        prediction,
        truth_primary,
        csec_frozen,
    )
    ablation_rows.append({
        "Environment": "Improved CSE-CIC-IDS2018",
        "Model": model_name,
        "Truth_Definition": "Primary_AttackShare_GE_50pct",
        **metrics,
    })

csec_structural_ablation = pd.DataFrame(ablation_rows)

preferred_columns = [
    "Environment",
    "Model",
    "Truth_Definition",
    "Flagged",
    "TP",
    "FP",
    "FN",
    "TN",
    "Precision",
    "Recall",
    "F1",
    "FPR",
    "Lift",
    "Unique_Flagged_Sources",
    "Unique_TP_Sources",
]
display_columns = [
    c for c in preferred_columns
    if c in csec_structural_ablation.columns
]

print("=" * 100)
print("IMPROVED CSE-CIC-IDS2018 — STRUCTURAL ABLATION BASELINES")
print("=" * 100)
print(
    "Calibration thresholds reused unchanged:\n"
    f"  COUNT P99 = {structural_thresholds['BV_COUNT_P99']:.6f}\n"
    f"  RATE  P99 = {structural_thresholds['BV_RATE_P99']:.6f}\n"
    f"  REP   P99 = {structural_thresholds['REP_RATIO_P99']:.6f}"
)
print()
print(
    csec_structural_ablation[display_columns]
    .to_string(index=False)
)

result_index = (
    csec_structural_ablation
    .set_index("Model")
)

diagnostic_pairs = [
    ("RATE_ONLY_P99", "S_BV", "SBV vs rate-only"),
    ("COUNT_ONLY_P99", "S_BV", "SBV vs count-only"),
    ("REP_ONLY_P99", "S_REP", "SREP vs repetition-only"),
]

diagnostic_rows = []
for baseline_name, operator_name, comparison in diagnostic_pairs:
    b = result_index.loc[baseline_name]
    o = result_index.loc[operator_name]

    diagnostic_rows.append({
        "Environment": "Improved CSE-CIC-IDS2018",
        "Comparison": comparison,
        "Baseline": baseline_name,
        "Operator": operator_name,
        "Delta_Flagged": int(o["Flagged"] - b["Flagged"]),
        "Delta_TP": int(o["TP"] - b["TP"]),
        "Delta_FP": int(o["FP"] - b["FP"]),
        "Delta_Precision": float(o["Precision"] - b["Precision"]),
        "Delta_Recall": float(o["Recall"] - b["Recall"]),
        "Delta_F1": float(o["F1"] - b["F1"]),
    })

csec_ablation_diagnostics = pd.DataFrame(diagnostic_rows)

print("\nConjunction-value diagnostics:")
print(
    csec_ablation_diagnostics
    .to_string(index=False)
)

csec_structural_ablation.to_csv(
    "/content/csecic2018_structural_ablation_baselines.csv",
    index=False,
)
csec_ablation_diagnostics.to_csv(
    "/content/csecic2018_structural_ablation_diagnostics.csv",
    index=False,
)

print("\nSaved:")
print("/content/csecic2018_structural_ablation_baselines.csv")
print("/content/csecic2018_structural_ablation_diagnostics.csv")


IMPROVED CSE-CIC-IDS2018 — STRUCTURAL ABLATION BASELINES
Calibration thresholds reused unchanged:
  COUNT P99 = 1902.370000
  RATE  P99 = 2.000000
  REP   P99 = 60.500000

             Environment          Model             Truth_Definition  Flagged  TP   FP  FN     TN  Precision   Recall       F1      FPR       Lift  Unique_Flagged_Sources  Unique_TP_Sources
Improved CSE-CIC-IDS2018 COUNT_ONLY_P99 Primary_AttackShare_GE_50pct     5697  81 5616  58 514592   0.014218 0.582734 0.027759 0.010796  53.225170                     542                 26
Improved CSE-CIC-IDS2018  RATE_ONLY_P99 Primary_AttackShare_GE_50pct     8439  50 8389  89 511819   0.005925 0.359712 0.011658 0.016126  22.179782                    6243                 16
Improved CSE-CIC-IDS2018   REP_ONLY_P99 Primary_AttackShare_GE_50pct     3913 125 3788  14 516420   0.031945 0.899281 0.061698 0.007282 119.585471                     434                 26
Improved CSE-CIC-IDS2018           S_BV Primary_AttackShare_GE_50pct

In [ ]:
# ============================================================
# COMPLETE COLAB RUNTIME ENVIRONMENT SNAPSHOT
# Saves all reproducibility information into ONE text file.
# ============================================================

import sys
import os
import platform
import subprocess
import datetime
import importlib
import psutil

OUTPUT_FILE = "/content/runtime_environment_full.txt"

def run_cmd(cmd):
    try:
        return subprocess.check_output(
            cmd,
            shell=True,
            stderr=subprocess.STDOUT,
            text=True
        ).strip()
    except Exception as e:
        return f"[Command failed: {e}]"

def pkg_version(name):
    try:
        module = importlib.import_module(name)
        return getattr(module, "__version__", "version attribute unavailable")
    except Exception as e:
        return f"not available ({e})"

now = datetime.datetime.now(datetime.timezone.utc)

vm = psutil.virtual_memory()
disk = psutil.disk_usage("/")

important_packages = [
    "numpy",
    "pandas",
    "scipy",
    "sklearn",
    "duckdb",
    "networkx",
    "matplotlib",
    "pyarrow",
    "psutil"
]

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    f.write("=" * 80 + "\n")
    f.write("COLAB RUNTIME ENVIRONMENT SNAPSHOT\n")
    f.write("=" * 80 + "\n\n")

    f.write("Captured (UTC): "
            + now.strftime("%Y-%m-%d %H:%M:%S UTC") + "\n\n")

    # --------------------------------------------------------
    # Python
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("PYTHON\n")
    f.write("=" * 80 + "\n")

    f.write(f"Python version:\n{sys.version}\n\n")
    f.write(f"Python executable:\n{sys.executable}\n\n")
    f.write(f"Implementation:\n{platform.python_implementation()}\n\n")

    # --------------------------------------------------------
    # Operating system
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("OPERATING SYSTEM\n")
    f.write("=" * 80 + "\n")

    f.write(f"platform.platform():\n{platform.platform()}\n\n")
    f.write(run_cmd("cat /etc/os-release") + "\n\n")
    f.write("Kernel:\n")
    f.write(run_cmd("uname -a") + "\n\n")

    # --------------------------------------------------------
    # CPU
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("CPU\n")
    f.write("=" * 80 + "\n")

    f.write(run_cmd("lscpu") + "\n\n")
    f.write(f"Logical CPU count: {psutil.cpu_count(logical=True)}\n")
    f.write(f"Physical CPU count: {psutil.cpu_count(logical=False)}\n\n")

    # --------------------------------------------------------
    # RAM
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("MEMORY\n")
    f.write("=" * 80 + "\n")

    f.write(f"Total RAM:     {vm.total / (1024**3):.2f} GB\n")
    f.write(f"Available RAM: {vm.available / (1024**3):.2f} GB\n")
    f.write(f"Used RAM:      {vm.used / (1024**3):.2f} GB\n")
    f.write(f"RAM usage:     {vm.percent:.1f}%\n\n")

    # --------------------------------------------------------
    # Disk
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("DISK\n")
    f.write("=" * 80 + "\n")

    f.write(f"Total disk: {disk.total / (1024**3):.2f} GB\n")
    f.write(f"Used disk:  {disk.used / (1024**3):.2f} GB\n")
    f.write(f"Free disk:  {disk.free / (1024**3):.2f} GB\n\n")

    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("GPU / ACCELERATOR\n")
    f.write("=" * 80 + "\n")

    gpu_info = run_cmd(
        "nvidia-smi --query-gpu="
        "name,memory.total,driver_version "
        "--format=csv,noheader 2>/dev/null"
    )

    if gpu_info:
        f.write(gpu_info + "\n\n")
        f.write("Full NVIDIA-SMI:\n")
        f.write(run_cmd("nvidia-smi") + "\n\n")
    else:
        f.write("No NVIDIA GPU detected or nvidia-smi unavailable.\n\n")

    # --------------------------------------------------------
    # Important package versions
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("IMPORTANT PACKAGE VERSIONS\n")
    f.write("=" * 80 + "\n")

    for package in important_packages:
        f.write(f"{package}: {pkg_version(package)}\n")

    f.write("\n")

    # --------------------------------------------------------
    # Google Colab
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("GOOGLE COLAB\n")
    f.write("=" * 80 + "\n")

    try:
        import google.colab
        f.write("Environment: Google Colab\n")
        f.write(
            "google.colab version: "
            + str(getattr(google.colab, "__version__", "not exposed"))
            + "\n"
        )
    except Exception:
        f.write("Google Colab module not detected.\n")

    f.write("\n")

    # --------------------------------------------------------
    # Environment variables relevant to computation
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("SELECTED ENVIRONMENT VARIABLES\n")
    f.write("=" * 80 + "\n")

    selected_env = [
        "COLAB_RELEASE_TAG",
        "COLAB_BACKEND_VERSION",
        "CUDA_VERSION",
        "LANG",
        "TZ"
    ]

    for var in selected_env:
        f.write(f"{var}: {os.environ.get(var, 'not set')}\n")

    f.write("\n")

    # --------------------------------------------------------
    # Complete pip package snapshot
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("COMPLETE PIP FREEZE\n")
    f.write("=" * 80 + "\n")

    f.write(run_cmd(f'"{sys.executable}" -m pip freeze') + "\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("END OF ENVIRONMENT SNAPSHOT\n")
    f.write("=" * 80 + "\n")

print("=" * 80)
print("Environment snapshot saved successfully.")
print(f"File: {OUTPUT_FILE}")
print(f"Size: {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")
print("=" * 80)

# Optional: automatically download the file from Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except Exception:
    print("Automatic download unavailable; download the file manually from /content.")

Environment snapshot saved successfully.
File: /content/runtime_environment_full.txt
Size: 18.8 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

for f in os.listdir("/content"):
    if "ablation" in f.lower():
        print(f)

csecic2018_structural_ablation_baselines.csv
csecic2018_structural_ablation_diagnostics.csv


In [ ]:
from google.colab import files

files.download('/content/csecic2018_structural_ablation_baselines.csv')
files.download('/content/csecic2018_structural_ablation_diagnostics.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>